---
title: NBA Possession Simulator Interactive Workspace
project: NBA Simulator
file_type: jupyter_notebook
status: active
purpose: Explore, demonstrate, and review the canonical possession simulator.
usage: Run the canonical import cell, then execute documented examples or custom scenarios.
last_updated: 2026-08-01
---

# NBA Possession Simulator — Interactive Workspace

## Purpose

Provide a Jupyter interface for exploring, demonstrating, and reviewing the canonical possession simulator. Production logic lives in `../nba_simulator/simulator_core.py`; this notebook imports that module instead of maintaining a second active implementation.

## How to use

1. Start Jupyter with either the project root or this `notebooks` folder as the working directory.
2. Run the canonical import cell near the top. It adds the project root to `sys.path` when needed.
3. Run the documented example cell or create teams and call the imported game, matchup, or season APIs.
4. Use the returned in-memory tables and audits for exploration; use `scripts/run_simulation.py` for repeatable command-line CSV exports.

## Inputs and outputs

Inputs are simulator team objects, `SimConfig`, optional `GameContext`, schedules, and versioned datasets under `../data`. Outputs include completed game dictionaries, possession/play logs, player and team tables, NBA-style box-score views, Monte Carlo summaries, and optional dated CSV files.

## Active versus historical cells

The canonical import and current example cells are the supported workflow. Later large code cells are preserved historical experiments from earlier notebook development; they are not production code and should not be run as an alternative engine. The original notebook is retained in `__Archive__`.


## ESPN Play-By-Play Coverage Audit

Reviewed ESPN game `401869393`: New York Knicks at Atlanta Hawks, final 140-89, with 477 play-by-play rows. After review, the simulator intentionally keeps a simplified event model: 2PT field goals, 3PT field goals, free throws, generic turnovers, rebounds, assists, steals, blocks, personal/shooting/offensive fouls, substitutions, timeouts, period boundaries, and end game.

Excluded by design: kicked ball, 3-second turnover, defensive goaltending, delay of game, double technical foul, coach challenge, personal take foul, shot clock turnover, out-of-bounds turnover variants, ejection, offensive foul turnover as a distinct label, technical foul, jumpball, and detailed shot subtypes.


## NBA.com-Style Box Score Outputs

The simulator now returns `game["box_score_views"]` with four completed-game views: `traditional`, `advanced`, `scoring`, and `defense`. Each view has `players` and `teams` rows modeled after NBA.com / NBA Stats box score endpoint schemas.

Current inputs are generated from placeholder player profiles (`usage`, shooting percentages, free throw rate, assist/turnover/rebound/steal/block/foul rates, defense, stamina, clutch, and minutes target). These placeholders drive possession probabilities now and can be replaced later with real player averages.


In [ ]:
# Canonical simulator implementation lives in nba_simulator/simulator_core.py.
from pathlib import Path
import sys
_project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(_project_root) not in sys.path: sys.path.insert(0, str(_project_root))
from nba_simulator.simulator_core import *


In [2]:
# Example: create two balanced sample rosters and run a detailed game.
# Replace these specs with real NBA player inputs as you collect/clean data.
home_specs = [
    ("LAL Lead Guard", "pg", {"mpg": 35, "usage": 29, "ast_rate": 0.36, "three_pct": 0.372, "clutch": 0.7}),
    ("LAL Scoring Wing", "wing", {"mpg": 34, "usage": 27, "two_pct": 0.548, "three_pct": 0.381, "defense": 0.5}),
    ("LAL 3&D Wing", "wing", {"mpg": 30, "usage": 16, "three_rate": 0.62, "three_pct": 0.392, "defense": 1.1}),
    ("LAL Stretch Forward", "forward", {"mpg": 31, "usage": 19, "three_rate": 0.43, "drb_rate": 0.19}),
    ("LAL Rim Big", "big", {"mpg": 32, "usage": 20, "two_pct": 0.635, "orb_rate": 0.12, "blk_rate": 0.065, "defense": 1.4}),
    ("LAL Sixth Man", "guard", {"mpg": 25, "usage": 24}),
    ("LAL Bench Wing", "wing", {"mpg": 18, "three_rate": 0.56}),
    ("LAL Backup Big", "big", {"mpg": 16, "orb_rate": 0.10}),
]

away_specs = [
    ("BOS Lead Guard", "pg", {"mpg": 34, "usage": 24, "ast_rate": 0.34, "defense": 0.8}),
    ("BOS Star Wing", "wing", {"mpg": 36, "usage": 30, "two_pct": 0.552, "three_pct": 0.374, "clutch": 0.8}),
    ("BOS Power Wing", "forward", {"mpg": 35, "usage": 26, "defense": 0.9, "drb_rate": 0.18}),
    ("BOS Spacer", "wing", {"mpg": 29, "usage": 15, "three_rate": 0.69, "three_pct": 0.401}),
    ("BOS Anchor Big", "big", {"mpg": 31, "two_pct": 0.650, "blk_rate": 0.060, "defense": 1.5}),
    ("BOS Bench Guard", "guard", {"mpg": 24, "usage": 21}),
    ("BOS Bench Forward", "forward", {"mpg": 18, "defense": 0.4}),
    ("BOS Backup Big", "big", {"mpg": 15, "orb_rate": 0.11}),
]

home_team = create_team("LAL", home_specs)
away_team = create_team("BOS", away_specs)

game = simulate_game_robust(home_team, away_team, SimConfig(seed=42))
print(game["final_score"], "winner:", game["winner"], "possessions:", game["possessions"])
print("\nTeam advanced stats")
for row in game["team_table"]:
    print(row)
print("\nTop player lines")
for row in sorted(game["player_table"], key=lambda r: (r["team"], -r["PTS"]))[:10]:
    print(row)
print("\nFirst 8 play-by-play events")
for event in game["play_by_play"][:8]:
    print(event)

print("\nBox score view keys")
print(game["box_score_views"].keys())
print("Traditional player row sample")
print(game["box_score_views"]["traditional"]["players"][0])
print("Advanced team row sample")
print(game["box_score_views"]["advanced"]["teams"][0])


{'LAL': 115, 'BOS': 131} winner: BOS possessions: 200

Team advanced stats
{'team': 'LAL', 'PTS': 115, 'POSS': 100, 'PACE': 96.0, 'ORTG': 115.0, 'DRTG': 131.0, 'eFG%': 0.582, 'TS%': 0.586, 'TOV%': 12.0, 'ORB%': 19.6, 'FTr': 0.152, 'AST%': 51.2, 'TIMEOUTS': 2, 'BONUS_FTA': 0, 'AVG_SPACING': 0.482, 'AVG_RIM_PRESSURE': 0.324, 'AVG_LINEUP_DEFENSE': 0.682, '+/-': -16}
{'team': 'BOS', 'PTS': 131, 'POSS': 100, 'PACE': 96.0, 'ORTG': 131.0, 'DRTG': 115.0, 'eFG%': 0.646, 'TS%': 0.687, 'TOV%': 11.0, 'ORB%': 14.3, 'FTr': 0.468, 'AST%': 61.9, 'TIMEOUTS': 3, 'BONUS_FTA': 0, 'AVG_SPACING': 0.471, 'AVG_RIM_PRESSURE': 0.327, 'AVG_LINEUP_DEFENSE': 0.723, '+/-': 16}

Top player lines
{'team': 'BOS', 'player': 'BOS Power Wing', 'MIN': 43.9, 'PTS': 33, 'REB': 10, 'AST': 1, 'STL': 0, 'BLK': 2, 'TOV': 5, 'PF': 1, '+/-': 16, 'FG': '11-19', '3P': '6-10', 'FT': '5-8', 'TS%': 0.733, 'eFG%': 0.737, 'USG%': 25.0, 'AST%': 2.4, 'TOV%': 20.0, 'REB%': 5.6, 'PeakFatigue': 0.26, 'LongStint': 7.4}
{'team': 'BOS', 'player

In [3]:
"""
basketball_sim.py
-----------------
Possession-by-possession basketball game simulator with fatigue, substitutions,
fouls, and simple coach AI. Supports:
 - Single-game detailed simulation with play-by-play log and box score.
 - Monte Carlo multi-game batch simulation for aggregated analytics.

Notes
-----
- This file focuses on readability & explainability: every function has a docstring
  and important internal logic is commented.
- No gameplay logic was changed; only documentation and comments were added.
"""

import random
import copy
from statistics import mean

# -------------------------------
# Config / Tunables
# -------------------------------
GAME_MINUTES = 48
SECONDS_PER_MIN = 60
GAME_SECONDS = GAME_MINUTES * SECONDS_PER_MIN
QUARTER_SECONDS = 12 * SECONDS_PER_MIN

# If None we simulate the game by time. Otherwise one could adapt to simulate
# a fixed number of possessions per team (not currently used elsewhere).
DEFAULT_POSSESSIONS_GOAL = None

# Substitution / fatigue tuning
SUB_CHECK_EVERY_POSSESSIONS = 4   # Check subs after roughly this many possessions
FATIGUE_PLAY_PENALTY = 0.006     # Shot% multiplier loss per unit of fatigue
FATIGUE_REB_PENALTY = 0.03       # Reduces rebound weight per fatigue unit
FATIGUE_DEF_PENALTY = 0.004      # Reduces defensive impact vs shooter per fatigue unit

# Coach AI thresholds: used to decide when coach goes into "aggressive" mode
COACH_AGGRESSIVE_DEFICIT = 8     # If trailing by this many points...
COACH_AGGRESSIVE_MINUTES_LEFT = 6  # ...and this many minutes remain -> go aggressive

# Baseline play type probabilities (normalized by pick_play_type)
PLAYTYPE_BASE = {
    'transition': 0.10,
    'pnr': 0.30,
    'iso': 0.20,
    'spotup': 0.25,
    'post': 0.15
}

# Targets / references (for tuning)
NBA_PACE_TARGET = 99
NBA_OFFRTG_TARGET = 114

# -------------------------------
# Helper: Create Player Template
# -------------------------------
def create_player(name, role='wing'):
    """
    Create a player template/dictionary with realistic (randomized) attributes.

    Parameters
    ----------
    name : str
        Identifier for player, e.g., "LAL_PG".
    role : str
        Archetype in {'pg','wing','big'}. Influences shooting splits and usage.

    Returns
    -------
    dict
        Player profile including basic shooting percentages, tendencies,
        fatigue/foul tracking, and a 'stats' sub-dictionary initialized to zeros.
    """
    # Role influences 2P/3P% and usage
    base_2p = random.uniform(0.47, 0.56) if role in ('pg', 'wing') else random.uniform(0.50, 0.60)
    base_3p = random.uniform(0.33, 0.40) if role != 'big' else random.uniform(0.20, 0.30)
    usage = random.uniform(0.8, 1.4) if role == 'pg' else random.uniform(0.7, 1.3)

    return {
        'name': name,
        'role': role,
        '2P%': base_2p,
        '3P%': base_3p,
        'FT%': random.uniform(0.72, 0.91),
        'usage': usage,                # proxy for how often this player is the shooter
        'TO%': random.uniform(0.09, 0.16),
        'clutch_boost': random.uniform(0.01, 0.04),  # small late-game accuracy bump
        'fatigue': 0.0,
        'fouls': 0,
        'disqualified': False,
        'sit_until': 0,                # timestamp when player is eligible from bench
        'ast_per_game': random.uniform(1.5, 8.0),
        'orb_per_game': random.uniform(0.5, 3.5),
        'drb_per_game': random.uniform(1.5, 8.0),
        'stats': {
            # box score counters
            'points': 0, 'fouls': 0, 'possessions': 0,
            'assists': 0, 'off_reb': 0, 'def_reb': 0,
            'tech_fouls': 0, 'flagrant_fouls': 0,
            'fgm': 0, 'fga': 0, '3pm': 0, '3pa': 0, 'ftm': 0, 'fta': 0
        }
    }

# -------------------------------
# Utility: swap players (substitution)
# -------------------------------
def swap_players(on_court, bench, idx_on, bench_player):
    """
    Swap a player from the bench into the on-court lineup.

    Parameters
    ----------
    on_court : list
        List of player dicts currently on the court.
    bench : list
        Bench player dicts.
    idx_on : int
        Index in on_court to replace.
    bench_player : dict
        Bench player object to bring in (must exist in bench).

    Returns
    -------
    bool
        True on success; False if bench_player not found in bench.
    """
    player_out = on_court[idx_on]
    if bench_player not in bench:
        return False
    bench.remove(bench_player)
    bench.append(player_out)
    on_court[idx_on] = bench_player
    return True

# -------------------------------
# Substitution Logic (improved)
# -------------------------------
def perform_subs(team_starters, team_bench, team_name, game_seconds_left, quarter_seconds,
                 current_quarter, score_diff, coach_style='balanced', log=None):
    """
    Evaluate and perform substitutions for one team.

    Logic:
    - Immediate sub if a player is disqualified or has fouled out (>=6 fouls).
    - Sub players with high fatigue or according to conservative coach strategy.
    - prioritizes eligible bench players (not disqualified, not sitting).

    Parameters
    ----------
    team_starters : list
        Current on-court players (first five positions).
    team_bench : list
        Bench players.
    team_name : str
        Team identifier (used for logging).
    game_seconds_left : int
        Seconds remaining in game (used for sit_until checks).
    quarter_seconds : int
        Seconds per quarter (not heavily used but provided).
    current_quarter : int or None
        If available, current quarter number (optional).
    score_diff : int
        (team_points - opponent_points) used for situational subs.
    coach_style : str
        One of {'balanced','aggressive','conservative'} affecting rest logic.
    log : list or None
        Optional list to append human-readable events.

    Returns
    -------
    int
        Number of substitutions made.
    """
    subs_made = 0

    # iterate over a shallow copy of starters so swapping while iterating is safe
    for i, p in enumerate(list(team_starters)):
        # 1) Fouled out or disqualified -> immediate replacement with best available
        if p.get('disqualified', False) or p['fouls'] >= 6:
            eligible = [b for b in team_bench if not b.get('disqualified', False) and b.get('sit_until', 0) <= game_seconds_left]
            if eligible:
                # choose the highest usage player (best playmaker/scorer)
                sub = max(eligible, key=lambda x: x['usage'])
                swap_players(team_starters, team_bench, i, sub)
                # give sub a small fatigue reprieve for coming in fresh
                sub['fatigue'] = max(0, sub['fatigue'] - 3)
                subs_made += 1
                if log: log.append(f"{team_name} - Sub for fouled out: {p['name']} -> {sub['name']}")

        # 2) Fatigue-driven substitution
        elif p['fatigue'] > 25 or (p['fatigue'] > 18 and coach_style == 'balanced'):
            eligible = [b for b in team_bench if not b.get('disqualified', False) and b.get('sit_until', 0) <= game_seconds_left]
            if eligible:
                # pick the freshest available sub
                sub = min(eligible, key=lambda x: x['fatigue'])
                swap_players(team_starters, team_bench, i, sub)
                subs_made += 1
                if log: log.append(f"{team_name} - Sub for fatigue: {p['name']} -> {sub['name']}")

        # 3) Situational rest subs (conservative coach, large lead, plenty of time)
        elif coach_style == 'conservative' and score_diff > 10 and game_seconds_left > 5 * 60:
            eligible = [b for b in team_bench if b.get('sit_until', 0) <= game_seconds_left]
            if eligible:
                sub = min(eligible, key=lambda x: x['fatigue'])
                swap_players(team_starters, team_bench, i, sub)
                subs_made += 1
                if log: log.append(f"{team_name} - Sub to rest (leading): {p['name']} -> {sub['name']}")

    return subs_made

# -------------------------------
# Rebounding Logic (improved weights + fatigue)
# -------------------------------
def get_rebound(off_court, def_court, is_offensive, log, team_name):
    """
    Select the rebounder and update stats.

    Uses role-driven rebound rates (orb_per_game / drb_per_game) reduced by fatigue.
    A small floor (0.01) prevents zero weights which would crash random.choices.
    """
    if is_offensive:
        # Offensive rebound probability based on player offensive rebound rate minus fatigue penalty
        reb_weights = [max(0.01, p['orb_per_game'] - p['fatigue'] * FATIGUE_REB_PENALTY) for p in off_court]
        rebounder = random.choices(off_court, weights=reb_weights, k=1)[0]
        rebounder['stats']['off_reb'] += 1
        log.append(f"{team_name} - {rebounder['name']} grabbed an offensive rebound!")
    else:
        # Defensive rebound probability based on defensive rebound rate minus fatigue penalty
        reb_weights = [max(0.01, p['drb_per_game'] - p['fatigue'] * FATIGUE_REB_PENALTY) for p in def_court]
        rebounder = random.choices(def_court, weights=reb_weights, k=1)[0]
        rebounder['stats']['def_reb'] += 1
        log.append(f"{team_name} - {rebounder['name']} grabbed a defensive rebound!")
    return rebounder

# -------------------------------
# Shot and Play Type Helpers
# -------------------------------
def pick_play_type(team, opponent, is_transition, game_seconds_left, score_diff, coach_aggressive=False):
    """
    Decide which play type the offense runs this possession.

    - Start from PLAYTYPE_BASE
    - If transition, boost transition weight and reduce others (empirical)
    - If coach_aggressive and trailing late, bias toward quicker/iso/spotup
    - Normalize and return a single play type string.

    Parameters
    ----------
    team : list
        Offensive players (unused currently but kept for potential extensions).
    opponent : list
        Defensive players (unused currently).
    is_transition : bool
        Whether the possession is a transition opportunity.
    game_seconds_left : int
        Seconds remaining — used for late-game adjustments.
    score_diff : int
        offense_points - defense_points (negative if trailing).
    coach_aggressive : bool
        If true, apply more aggressive play adjustments.
    """
    probs = PLAYTYPE_BASE.copy()

    if is_transition:
        # Favor transition heavily during transition events
        probs = {k: v * (1.5 if k == 'transition' else 0.5) for k, v in probs.items()}

    # Coach behavior: trailing & time remaining -> more quick plays and 3s
    if coach_aggressive and game_seconds_left > COACH_AGGRESSIVE_MINUTES_LEFT * 60 and score_diff < -COACH_AGGRESSIVE_DEFICIT:
        probs['transition'] += 0.05
        probs['iso'] += 0.05
        probs['spotup'] += 0.03

    total = sum(probs.values())
    for k in probs:
        probs[k] /= total

    choices, weights = zip(*probs.items())
    return random.choices(choices, weights=weights, k=1)[0]

def shot_modifier_by_play(play_type):
    """
    Return multipliers for shot efficiency and rebound effort by play type.

    These are empirical multipliers tuned to simulate differing shot quality
    across play types (e.g., post plays increase 2P efficiency; spot-up favors 3P).
    """
    if play_type == 'transition':
        return {'2P_mult': 1.05, '3P_mult': 0.9, 'rebound_effort': 0.9}
    if play_type == 'pnr':
        return {'2P_mult': 1.02, '3P_mult': 1.03, 'rebound_effort': 1.0}
    if play_type == 'iso':
        return {'2P_mult': 0.95, '3P_mult': 1.05, 'rebound_effort': 0.95}
    if play_type == 'spotup':
        return {'2P_mult': 0.95, '3P_mult': 1.10, 'rebound_effort': 0.9}
    if play_type == 'post':
        return {'2P_mult': 1.10, '3P_mult': 0.7, 'rebound_effort': 1.1}
    # default neutral modifier
    return {'2P_mult': 1.0, '3P_mult': 1.0, 'rebound_effort': 1.0}

# -------------------------------
# Simulate a Possession (full)
# -------------------------------
def simulate_possession(off_court, def_court, game_seconds_left, team_fouls, is_home, log,
                        offense_team_name, defense_team_name, team_def_rating, opponent_def_rating,
                        coach_aggressive=False):
    """
    Simulate one possession for the offense (off_court vs def_court).

    Parameters
    ----------
    off_court : list
        Offensive players on court.
    def_court : list
        Defensive players on court.
    game_seconds_left : int
        Seconds left in the game (affects clutch behavior).
    team_fouls : dict
        Dictionary tracking fouls for teams; also used to pass score_diff in this code.
    is_home : bool
        Whether the offense is the home team (small home shooting bonus applied).
    log : list
        Play-by-play list where textual events are appended.
    offense_team_name : str
        Offense team identifier (used in logs).
    defense_team_name : str
        Defense team identifier (used in logs and team_fouls updates).
    team_def_rating : float
        Defensive rating multiplier for the defense (1.0 baseline; <1 better D).
    opponent_def_rating : float
        Defensive rating of the opponent (kept for future use).
    coach_aggressive : bool
        Whether the coach is in aggressive mode (shorter possessions, more risk).

    Returns
    -------
    tuple
        (points_scored, possession_time_seconds, rebound_team_list_or_None, play_type_str)
    """
    # Small baseline probability that the possession is a transition event
    is_transition = random.random() < 0.08
    # pick shooter weighted by usage
    shooter = random.choices(off_court, weights=[p['usage'] for p in off_court], k=1)[0]

    # play selection
    play_type = pick_play_type(off_court, def_court, is_transition, game_seconds_left,
                               score_diff=team_fouls.get('score_diff', 0), coach_aggressive=coach_aggressive)
    play_mod = shot_modifier_by_play(play_type)

    # fatigue penalties (capped)
    fatigue_penalty = min(0.25, shooter['fatigue'] * FATIGUE_PLAY_PENALTY)
    fatigue_def_pen = min(0.25, shooter['fatigue'] * FATIGUE_DEF_PENALTY)

    # small clutch/home modifiers
    clutch_bonus = shooter.get('clutch_boost', 0.0) if game_seconds_left <= 120 else 0.0
    home_bonus = 0.02 if is_home else 0.0

    # team defense multipliers (lower <1 means better defense)
    team_def_factor = team_def_rating
    opp_def_factor = opponent_def_rating  # currently unused in decision but passed for extensibility

    # Late-game 3-point bias when trailing
    shooter_3pt_skill = shooter['3P%']
    if game_seconds_left < 2 * 60 and team_fouls.get('score_diff', 0) < 0:
        three_bias = 0.25
    else:
        three_bias = 0.0

    # Build outcome choice weights: 2P, 3P, turnover, foul
    base_two_weight = 0.50 * play_mod['2P_mult']
    base_three_weight = (0.20 + (0.5 * shooter_3pt_skill)) * play_mod['3P_mult']
    if shooter['role'] == 'big':
        # bigs take fewer threes
        base_three_weight *= 0.5
    turnover_weight = shooter['TO%'] * 1.0
    foul_weight = 0.14

    base_three_weight += three_bias

    # ensure no value is zero (prevents random.choices errors) and pick outcome
    weights = [
        max(0.01, base_two_weight),
        max(0.01, base_three_weight),
        max(0.01, turnover_weight),
        max(0.01, foul_weight)
    ]
    outcome = random.choices(['2P', '3P', 'TO', 'Foul'], weights=weights, k=1)[0]

    # possession duration (roughly) depends on play type
    base_possession_time = {
        'transition': random.randint(5, 10),
        'pnr': random.randint(8, 18),
        'iso': random.randint(7, 18),
        'spotup': random.randint(6, 16),
        'post': random.randint(8, 20)
    }.get(play_type, random.randint(8, 16))

    # If coach is aggressive (fast), shorten possessions slightly
    if coach_aggressive:
        base_possession_time = max(4, int(base_possession_time * 0.9))

    possession_time = base_possession_time
    points = 0
    rebound_team = None

    # Handle made/missed shots
    if outcome in ['2P', '3P']:
        base_prob = shooter['2P%'] if outcome == '2P' else shooter['3P%']
        # apply play modifiers, fatigue and defense adjustments; clamp probability
        effective_prob = base_prob * (play_mod['2P_mult'] if outcome == '2P' else play_mod['3P_mult'])
        effective_prob = effective_prob - fatigue_penalty - (team_def_factor - 1.0) * 0.02 + clutch_bonus + home_bonus
        effective_prob = max(0.03, min(0.95, effective_prob))
        made = random.random() < effective_prob

        # update shooter attempt counters
        shooter['stats']['fga'] += 1
        if outcome == '3P':
            shooter['stats']['3pa'] += 1

        if made:
            pts = 2 if outcome == '2P' else 3
            points += pts
            shooter['stats']['fgm'] += 1
            if outcome == '3P':
                shooter['stats']['3pm'] += 1
            shooter['stats']['points'] += pts

            # chance for assist credited to a teammate
            eligible_passers = [p for p in off_court if p['name'] != shooter['name']]
            if eligible_passers and random.random() < 0.28:  # baseline assist chance
                assister = random.choices(eligible_passers, weights=[p['ast_per_game'] for p in eligible_passers], k=1)[0]
                assister['stats']['assists'] += 1
                log.append(f"{offense_team_name} - {assister['name']} assisted {shooter['name']} ({play_type})")
        else:
            # missed shot -> contested rebound. off_reb_prob influenced by play type
            off_reb_prob = 0.28 * play_mod.get('rebound_effort', 1.0)
            is_offensive_reb = random.random() < off_reb_prob
            rebound_team = off_court if is_offensive_reb else def_court
            get_rebound(off_court, def_court, is_offensive_reb, log, offense_team_name if is_offensive_reb else defense_team_name)
            possession_time += random.randint(2, 5)

    # Turnover: immediate change of possession (or live turnover)
    elif outcome == 'TO':
        rebound_team = def_court
        # quick possession time for turnover
        possession_time = random.randint(3, 8)

    # Foul handling (many cases: technical, flagrant, shooting, regular)
    elif outcome == 'Foul':
        defender = random.choice(def_court)
        defender['fouls'] += 1
        defender['stats']['fouls'] = defender['fouls']

        foul_type = random.choices(['regular', 'shooting', 'technical', 'flagrant'],
                                   weights=[0.75, 0.18, 0.05, 0.02], k=1)[0]

        # If foul counts toward team fouls, increment team_fouls
        count_toward = foul_type in ('regular', 'shooting')
        if count_toward:
            team_fouls[defense_team_name] += 1

        # Free throw probability base (clamped between 0.5 and 0.98)
        prob_base = min(0.98, max(0.5, shooter['FT%'] - fatigue_penalty + clutch_bonus + home_bonus))

        if foul_type == 'technical':
            # single FT, team retains possession
            made = random.random() < prob_base
            shooter['stats']['ftm'] += int(made)
            shooter['stats']['fta'] += 1
            shooter['stats']['points'] += int(made)
            points += int(made)
            defender['stats']['tech_fouls'] += 1
            rebound_team = off_court

        elif foul_type == 'flagrant':
            # two FTs and possession retained
            fts = [random.random() < prob_base for _ in range(2)]
            made = sum(1 for m in fts if m)
            shooter['stats']['fta'] += 2
            shooter['stats']['ftm'] += made
            shooter['stats']['points'] += made
            points += made
            defender['stats']['flagrant_fouls'] += 1
            rebound_team = off_court

        elif foul_type == 'shooting':
            # Shooting foul: determine if and-1, or number of FTs
            shot_type = random.choices(['2P', '3P'], weights=[0.8, 0.2], k=1)[0]
            if random.random() < 0.25:
                # and-1: shooter is credited with made shot + 1 FT
                base_made_pts = 2 if shot_type == '2P' else 3
                shooter['stats']['points'] += base_made_pts
                shooter['stats']['fgm'] += 1
                shooter['stats']['fga'] += 1
                if shot_type == '3P':
                    shooter['stats']['3pm'] += 1
                    shooter['stats']['3pa'] += 1
                made_ft = random.random() < prob_base
                shooter['stats']['fta'] += 1
                shooter['stats']['ftm'] += int(made_ft)
                shooter['stats']['points'] += int(made_ft)
                points += base_made_pts + int(made_ft)
            else:
                # Missed on shot -> 2 or 3 free throws
                ft_attempts = 2 if shot_type == '2P' else 3
                fts = [random.random() < prob_base for _ in range(ft_attempts)]
                made = sum(1 for m in fts if m)
                shooter['stats']['fta'] += ft_attempts
                shooter['stats']['ftm'] += made
                shooter['stats']['points'] += made
                points += made
                # if any FT missed, rebound battle follows
                if any(not m for m in fts):
                    off_reb_prob = 0.32
                    is_offensive_reb = random.random() < off_reb_prob
                    rebound_team = off_court if is_offensive_reb else def_court
                    get_rebound(off_court, def_court, is_offensive_reb, log, offense_team_name if is_offensive_reb else defense_team_name)
                    possession_time += 2

        else:
            # Regular non-shooting foul: if defense in bonus -> FTs, else possession usually continues
            bonus = team_fouls[defense_team_name] >= 5
            if bonus:
                fts = [random.random() < prob_base for _ in range(2)]
                made = sum(1 for m in fts if m)
                shooter['stats']['fta'] += 2
                shooter['stats']['ftm'] += made
                shooter['stats']['points'] += made
                points += made
            else:
                # Ball stays with offense (shorter possession due to stoppage)
                possession_time = random.randint(3, 7)
                rebound_team = off_court

        # Potential foul out
        if defender['fouls'] >= 6:
            defender['disqualified'] = True
            log.append(f"{defense_team_name} - {defender['name']} has fouled out!")

    # Final bookkeeping: possessions and fatigue changes
    shooter['stats']['possessions'] += 1
    # Shooting possessions are more fatiguing; transition adds more
    shooter['fatigue'] += 1 + (2 if play_type == 'transition' else 0)
    # defenders gain small fatigue each possession
    for p in def_court:
        p['fatigue'] += 0.2

    return points, possession_time, rebound_team, play_type

# -------------------------------
# Game Simulation
# -------------------------------
def simulate_game(home_team_name, home_roster, away_team_name, away_roster,
                  home_def_rating=1.0, away_def_rating=1.0,
                  coach_home='balanced', coach_away='balanced', verbose=False):
    """
    Simulate one full basketball game.

    Parameters
    ----------
    home_team_name : str
        Home team identifier.
    home_roster : list
        List of player dicts for home team (first 5 used as starters).
    away_team_name : str
        Away team identifier.
    away_roster : list
        List of player dicts for away team.
    home_def_rating, away_def_rating : float
        Multiplicative defensive rating modifiers (1.0 baseline).
    coach_home, coach_away : str
        Coaching style; affects substitution and pace.
    verbose : bool
        If True, print summary at end.

    Returns
    -------
    dict
        Contains final score, team stats, player stats, play-by-play log, pace, and possessions.
    """
    # Work on deep copies so original rosters remain unchanged by simulation
    home = copy.deepcopy(home_roster)
    away = copy.deepcopy(away_roster)

    # starters are first five elements (simple convention)
    home_starters, home_bench = home[:5], home[5:]
    away_starters, away_bench = away[:5], away[5:]

    # Reset per-game state for every player
    for p in home + away:
        p['fatigue'] = 0.0
        p['fouls'] = 0
        p['disqualified'] = False
        p['sit_until'] = 0
        p['stats'] = {
            'points': 0, 'fouls': 0, 'possessions': 0,
            'assists': 0, 'off_reb': 0, 'def_reb': 0,
            'tech_fouls': 0, 'flagrant_fouls': 0,
            'fgm': 0, 'fga': 0, '3pm': 0, '3pa': 0, 'ftm': 0, 'fta': 0
        }

    # Game state initialization
    score = {home_team_name: 0, away_team_name: 0}
    team_fouls = {home_team_name: 0, away_team_name: 0}
    possession_team = home_team_name  # simple kickoff: home team starts with ball
    time_left = GAME_SECONDS
    log = []
    possessions = 0

    # track possessions per team to compute pace and OffRtg
    team_possessions = {home_team_name: 0, away_team_name: 0}

    # coach state includes style and a boolean aggressive flag computed during game
    coach_state = {
        home_team_name: {'style': coach_home, 'aggressive': False},
        away_team_name: {'style': coach_away, 'aggressive': False}
    }

    # Main time-driven simulation loop
    while time_left > 0:
        # Update coach aggression flag based on scoreboard and remaining time
        for tname in (home_team_name, away_team_name):
            other = away_team_name if tname == home_team_name else home_team_name
            lead = score[tname] - score[other]
            if lead < -COACH_AGGRESSIVE_DEFICIT and time_left <= COACH_AGGRESSIVE_MINUTES_LEFT * 60:
                coach_state[tname]['aggressive'] = True
            else:
                coach_state[tname]['aggressive'] = False

        # Determine which team is on offense and set local variables accordingly
        if possession_team == home_team_name:
            off_court = home_starters
            def_court = away_starters
            is_home = True
            off_def_rating = away_def_rating
            def_def_rating = home_def_rating
            offense_name = home_team_name
            defense_name = away_team_name
        else:
            off_court = away_starters
            def_court = home_starters
            is_home = False
            off_def_rating = home_def_rating
            def_def_rating = away_def_rating
            offense_name = away_team_name
            defense_name = home_team_name

        # Pass score difference through team_fouls dict for compatibility with simulate_possession
        team_fouls['score_diff'] = score[offense_name] - score[defense_name]

        # Simulate the possession
        pts, dur, rebound_team, play_type = simulate_possession(
            off_court, def_court, game_seconds_left=time_left, team_fouls=team_fouls,
            is_home=is_home, log=log, offense_team_name=offense_name, defense_team_name=defense_name,
            team_def_rating=(off_def_rating), opponent_def_rating=(def_def_rating),
            coach_aggressive=coach_state[offense_name]['aggressive']
        )

        # Apply points to scoreboard for the team that had possession
        score[possession_team] += pts
        team_possessions[possession_team] += 1
        possessions += 1

        # Time advances by the possession duration; ensure non-negative
        time_left -= dur
        if time_left < 0:
            time_left = 0

        # Periodic substitutions check
        if possessions % SUB_CHECK_EVERY_POSSESSIONS == 0:
            home_diff = score[home_team_name] - score[away_team_name]
            away_diff = -home_diff
            perform_subs(home_starters, home_bench, home_team_name, time_left, QUARTER_SECONDS, None, home_diff, coach_style=coach_state[home_team_name]['style'], log=log)
            perform_subs(away_starters, away_bench, away_team_name, time_left, QUARTER_SECONDS, None, away_diff, coach_style=coach_state[away_team_name]['style'], log=log)

        # Possession change logic based on rebounds or turnovers:
        # - If rebound_team is None => defensive rebound (possession flips)
        # - If rebound_team equals defensive court list => defensive rebound => flip
        # - Else offense retained the ball
        if rebound_team is None:
            possession_team = away_team_name if possession_team == home_team_name else home_team_name
        else:
            if rebound_team == def_court:
                possession_team = away_team_name if possession_team == home_team_name else home_team_name
            else:
                # offense retains; no change to possession_team
                pass

        # Quarter break logic: when remaining time is exact multiple of quarter length, reset fouls and apply rest
        elapsed = GAME_SECONDS - time_left
        if time_left % QUARTER_SECONDS == 0 and time_left != GAME_SECONDS:
            # Reset team fouls for the new quarter
            team_fouls = {home_team_name: 0, away_team_name: 0}
            # Give players a little fatigue relief when quarter ends
            for p in home_starters + away_starters + home_bench + away_bench:
                p['fatigue'] = max(0, p['fatigue'] - 2)
                p['sit_until'] = 0
            log.append(f"--- End of Q{int(elapsed // QUARTER_SECONDS)} - Fouls reset and small rest ---")

    # End of regulation; compile team and player analytics
    def compile_team_stats(team_name, starters, bench):
        """
        Aggregate per-player stats into team-level box score and simple advanced metrics.
        """
        players = starters + bench
        pts = sum(p['stats']['points'] for p in players)
        fgm = sum(p['stats'].get('fgm', 0) for p in players)
        fga = sum(p['stats'].get('fga', 0) for p in players)
        threem = sum(p['stats'].get('3pm', 0) for p in players)
        threea = sum(p['stats'].get('3pa', 0) for p in players)
        ftm = sum(p['stats'].get('ftm', 0) for p in players)
        fta = sum(p['stats'].get('fta', 0) for p in players)
        ast = sum(p['stats']['assists'] for p in players)
        orb = sum(p['stats']['off_reb'] for p in players)
        drb = sum(p['stats']['def_reb'] for p in players)
        possessions_count = team_possessions[team_name] if team_name in team_possessions else 0

        # Safely compute percentages with zero-division guards
        fg_pct = fgm / fga if fga > 0 else 0
        three_pct = threem / threea if threea > 0 else 0
        efg = (fgm + 0.5 * threem) / fga if fga > 0 else 0
        off_rtg = (pts / possessions_count * 100) if possessions_count > 0 else 0

        return {
            'pts': pts, 'fgm': fgm, 'fga': fga, 'fg%': fg_pct,
            '3pm': threem, '3pa': threea, '3p%': three_pct,
            'ftm': ftm, 'fta': fta, 'ast': ast, 'orb': orb, 'drb': drb,
            'eFG%': efg, 'OffRtg': off_rtg, 'possessions': possessions_count
        }

    home_stats = compile_team_stats(home_team_name, home_starters, home_bench)
    away_stats = compile_team_stats(away_team_name, away_starters, away_bench)

    # Pace estimation: convert total possessions to 48-minute pace baseline
    total_poss = team_possessions[home_team_name] + team_possessions[away_team_name]
    pace = total_poss * (48 / GAME_MINUTES) if GAME_MINUTES > 0 else 0

    # Flatten player stats for convenient box-score output
    players_final = {p['name']: p['stats'] for p in home + away}

    # Optional verbose prints for debugging
    if verbose:
        print(f"FINAL SCORE: {home_team_name} {score[home_team_name]} - {away_team_name} {score[away_team_name]}")
        print(f"Home team stats: {home_stats}")
        print(f"Away team stats: {away_stats}")
        print(f"Pace estimate: {pace:.1f}, Total possessions: {total_poss}")

    return {
        'score': score,
        'home_stats': home_stats,
        'away_stats': away_stats,
        'players': players_final,
        'log': log,
        'pace': pace,
        'team_possessions': team_possessions
    }

# -------------------------------
# Monte Carlo wrapper (optional)
# -------------------------------
def monte_carlo(home_team_name, away_team_name, home_roster, away_roster, n=10, verbose=False):
    """
    Run multiple independent game simulations and return the raw results list.

    Each game uses slightly randomized defensive ratings and randomly-picked
    coach styles to add variety to Monte Carlo runs.

    Returns
    -------
    list
        List of result dicts (one per simulated game).
    """
    results = []
    for i in range(n):
        res = simulate_game(
            home_team_name, home_roster, away_team_name, away_roster,
            home_def_rating=random.uniform(0.96, 1.04),
            away_def_rating=random.uniform(0.96, 1.04),
            coach_home=random.choice(['balanced', 'aggressive', 'conservative']),
            coach_away=random.choice(['balanced', 'aggressive', 'conservative']),
            verbose=False
        )
        results.append(res)

    # Helper aggregator (kept for potential future use)
    def agg(key):
        # If the result contains the expected dict at key, average the 'pts'
        return mean([r[key]['pts'] if key in r else 0 for r in results]) if isinstance(results[0][key], dict) else None

    if verbose:
        for i, r in enumerate(results):
            print(f"Game {i+1}: {r['score'][home_team_name]} - {r['score'][away_team_name]}; Pace {r['pace']:.1f}")

    return results

# -------------------------------
# Single Game Simulation Wrapper
# -------------------------------
def simulate_single_game(home_team_name, away_team_name, home_roster, away_roster,
                         home_def_rating=1.0, away_def_rating=1.0,
                         coach_home='balanced', coach_away='balanced'):
    """
    Convenience wrapper to run a single detailed simulation and print a readable
    play-by-play, box score, and advanced team stats.

    This function calls simulate_game() and then formats the returned data for
    immediate human consumption (console output).
    """
    print(f"=== Simulating {home_team_name} vs {away_team_name} (Single Game Mode) ===\n")
    result = simulate_game(
        home_team_name, home_roster, away_team_name, away_roster,
        home_def_rating=home_def_rating,
        away_def_rating=away_def_rating,
        coach_home=coach_home,
        coach_away=coach_away,
        verbose=False
    )

    # PLAY-BY-PLAY
    print("\n--- PLAY-BY-PLAY LOG ---")
    for line in result["log"]:
        print(line)
    print("\n--- END OF GAME ---")

    # SUMMARY
    print(f"\nFINAL SCORE: {home_team_name} {result['score'][home_team_name]} - {away_team_name} {result['score'][away_team_name]}")
    print(f"Pace: {result['pace']:.1f}  |  Possessions: {result['team_possessions']}")
    print("\nTEAM STATS:")
    print(f"{home_team_name} - {result['home_stats']}")
    print(f"{away_team_name} - {result['away_stats']}")

    # BOX SCORE (player-by-player)
    print("\n--- BOX SCORE ---")
    all_players = result['players']
    # Sort players by points descending for readable output
    for name, stats in sorted(all_players.items(), key=lambda kv: kv[1].get('points', 0), reverse=True):
        # Format counts safely (some fields default to 0)
        print(f"{name:10s} | {stats.get('points',0):2d} pts | FGM/FGA {stats.get('fgm',0)}/{stats.get('fga',0)} | "
              f"3P {stats.get('3pm',0)}/{stats.get('3pa',0)} | FT {stats.get('ftm',0)}/{stats.get('fta',0)} | AST {stats.get('assists',0)} | "
              f"REB {stats.get('off_reb',0)+stats.get('def_reb',0)} ({stats.get('off_reb',0)}/{stats.get('def_reb',0)}) | "
              f"Fouls {stats.get('fouls',0)}")

    # Top scorers summary
    top_scorers = sorted(all_players.items(), key=lambda kv: kv[1].get('points', 0), reverse=True)[:5]
    print("\nTop Scorers:")
    for name, stats in top_scorers:
        print(f" - {name}: {stats.get('points',0)} pts, {stats.get('fgm',0)}/{stats.get('fga',0)} FG, {stats.get('3pm',0)}/{stats.get('3pa',0)} 3P")

    # Advanced Team Stats (eFG% and OffRtg)
    print("\nAdvanced Stats:")
    for team, stats in zip([home_team_name, away_team_name], [result['home_stats'], result['away_stats']]):
        print(f"{team}: eFG% {stats['eFG%']:.3f}, OffRtg {stats['OffRtg']:.1f}")

    return result

# -------------------------------
# Multi-Game Simulation Wrapper
# -------------------------------
def simulate_multiple_games(home_team_name, away_team_name, home_roster, away_roster,
                            n=50, verbose=False):
    """
    Run many games (Monte Carlo) without verbose play-by-play and aggregate
    team averages, win-loss record, and pace.

    Parameters
    ----------
    home_team_name, away_team_name : str
        Team identifiers.
    home_roster, away_roster : list
        Player lists for each team.
    n : int
        Number of simulated games to run.
    verbose : bool
        If True, prints each simulated game's score and pace.

    Returns
    -------
    dict
        Summary including average team stats, pace, and series wins.
    """
    print(f"=== Simulating {n} games between {home_team_name} and {away_team_name} (Batch Mode) ===")
    results = monte_carlo(home_team_name, away_team_name, home_roster, away_roster, n=n, verbose=False)

    # Aggregate team-level points and compute wins
    home_pts = [r['score'][home_team_name] for r in results]
    away_pts = [r['score'][away_team_name] for r in results]
    home_wins = sum(1 for h, a in zip(home_pts, away_pts) if h > a)
    away_wins = n - home_wins
    avg_pace = mean(r['pace'] for r in results)

    def avg_team_stats(key):
        # Compute averages across simulation runs for key (home_stats/away_stats)
        return {
            'pts': mean(r[key]['pts'] for r in results),
            'fg%': mean(r[key]['fg%'] for r in results),
            '3p%': mean(r[key]['3p%'] for r in results),
            'eFG%': mean(r[key]['eFG%'] for r in results),
            'OffRtg': mean(r[key]['OffRtg'] for r in results),
        }

    home_avg = avg_team_stats('home_stats')
    away_avg = avg_team_stats('away_stats')

    print("\n--- SERIES RESULTS ---")
    print(f"{home_team_name}: {home_wins}-{away_wins} record over {n} games")
    print(f"Average Pace: {avg_pace:.1f}\n")
    print(f"{home_team_name} Avg Stats: {home_avg}")
    print(f"{away_team_name} Avg Stats: {away_avg}")

    return {
        'home_avg': home_avg,
        'away_avg': away_avg,
        'pace': avg_pace,
        'home_wins': home_wins,
        'away_wins': away_wins
    }

# -------------------------------
# Example Usage
# -------------------------------
if __name__ == "__main__":
    # Build small sample rosters; first five are starters as per convention above
    home_roster = [create_player(f"LAL_{r}", role=random.choice(['pg', 'wing', 'big'])) for r in
                   ["PG", "W1", "W2", "B1", "B2", "BEN1", "BEN2", "BEN3", "BEN4", "BEN5"]]
    away_roster = [create_player(f"BOS_{r}", role=random.choice(['pg', 'wing', 'big'])) for r in
                   ["PG", "W1", "W2", "B1", "B2", "BEN1", "BEN2", "BEN3", "BEN4", "BEN5"]]

    # Single detailed game
    result = simulate_single_game("LAL", "BOS", home_roster, away_roster,
                                  home_def_rating=0.98, away_def_rating=1.02,
                                  coach_home='balanced', coach_away='aggressive')

    # Multi-game Monte Carlo run (20 sims)
    print("\n\n--- MULTI-GAME ANALYSIS ---")
    series = simulate_multiple_games("LAL", "BOS", home_roster, away_roster, n=20)


=== Simulating LAL vs BOS (Single Game Mode) ===


--- PLAY-BY-PLAY LOG ---
LAL - LAL_B2 assisted LAL_PG (spotup)
BOS - BOS_W2 grabbed a defensive rebound!
LAL - LAL_PG assisted LAL_W2 (pnr)
LAL - LAL_W2 grabbed a defensive rebound!
LAL - LAL_W2 grabbed a defensive rebound!
LAL - LAL_B2 grabbed a defensive rebound!
BOS - BOS_W1 grabbed a defensive rebound!
BOS - BOS_W1 grabbed a defensive rebound!
BOS - BOS_PG grabbed a defensive rebound!
BOS - BOS_PG grabbed an offensive rebound!
LAL - LAL_W1 grabbed a defensive rebound!
BOS - BOS_PG grabbed a defensive rebound!
LAL - LAL_W2 grabbed a defensive rebound!
BOS - BOS_B2 grabbed a defensive rebound!
BOS - BOS_W1 grabbed a defensive rebound!
BOS - BOS_W2 grabbed a defensive rebound!
LAL - LAL_B1 grabbed a defensive rebound!
BOS - BOS_W1 grabbed a defensive rebound!
LAL - LAL_PG grabbed an offensive rebound!
LAL - LAL_PG assisted LAL_W1 (transition)
LAL - LAL_B2 grabbed a defensive rebound!
LAL - LAL_PG assisted LAL_W1 (spotup)
LAL - LAL_W2 

In [13]:
import random
import copy
from statistics import mean

# -------------------------------
# Config / Tunables
# -------------------------------
GAME_MINUTES = 48
SECONDS_PER_MIN = 60
GAME_SECONDS = GAME_MINUTES * SECONDS_PER_MIN
QUARTER_SECONDS = 12 * SECONDS_PER_MIN
SUB_CHECK_EVERY_POSSESSIONS = 4
FATIGUE_PLAY_PENALTY = 0.006
FATIGUE_REB_PENALTY = 0.03
FATIGUE_DEF_PENALTY = 0.004
COACH_AGGRESSIVE_DEFICIT = 8
COACH_AGGRESSIVE_MINUTES_LEFT = 6

PLAYTYPE_BASE = {
    'transition': 0.10,
    'pnr': 0.30,
    'iso': 0.20,
    'spotup': 0.25,
    'post': 0.15
}

# -------------------------------
# Player creation with advanced metrics
# -------------------------------
def create_player(name, role='wing', mpg=30, stats_input=None):
    """
    Create a player profile with advanced stats.

    stats_input can be a dict with keys:
      '2PA', '2P%', '3PA', '3P%', 'FTA', 'FT%', 'ORB', 'DRB', 'AST', 'STL',
      'BLK', 'TO', 'PF', 'usage%'
    """
    if stats_input is None:
        stats_input = {}
    return {
        'name': name,
        'role': role,
        'mpg': mpg,
        '2PA': stats_input.get('2PA', random.uniform(5,15)),
        '2P%': stats_input.get('2P%', random.uniform(0.45,0.55)),
        '3PA': stats_input.get('3PA', random.uniform(0,6)),
        '3P%': stats_input.get('3P%', random.uniform(0.33,0.38)),
        'FTA': stats_input.get('FTA', random.uniform(1,5)),
        'FT%': stats_input.get('FT%', random.uniform(0.7,0.9)),
        'ORB': stats_input.get('ORB', random.uniform(0.5,3)),
        'DRB': stats_input.get('DRB', random.uniform(1.5,8)),
        'AST': stats_input.get('AST', random.uniform(1,8)),
        'STL': stats_input.get('STL', random.uniform(0,2)),
        'BLK': stats_input.get('BLK', random.uniform(0,2)),
        'TO': stats_input.get('TO', random.uniform(1,4)),
        'PF': stats_input.get('PF', random.uniform(1,4)),
        'usage': stats_input.get('usage', random.uniform(15,30)),
        'fatigue': 0.0,
        'fouls': 0,
        'disqualified': False,
        'sit_until': 0,
        'stats': {
            'points':0,'fouls':0,'possessions':0,'assists':0,'off_reb':0,'def_reb':0,
            'steals':0,'blocks':0,'turnovers':0,'fgm':0,'fga':0,'3pm':0,'3pa':0,'ftm':0,'fta':0
        }
    }

# -------------------------------
# Helper: swap players
# -------------------------------
def swap_players(on_court, bench, idx_on, bench_player):
    player_out = on_court[idx_on]
    if bench_player not in bench:
        return False
    bench.remove(bench_player)
    bench.append(player_out)
    on_court[idx_on] = bench_player
    return True

# -------------------------------
# Substitution logic
# -------------------------------
def perform_subs(starters, bench, team_name, game_seconds_left, score_diff, coach_style='balanced', log=None):
    subs_made = 0
    for i, p in enumerate(list(starters)):
        if p.get('disqualified', False) or p['fouls'] >= 6:
            eligible = [b for b in bench if not b.get('disqualified', False) and b.get('sit_until',0)<=game_seconds_left]
            if eligible:
                sub = max(eligible, key=lambda x: x['usage'])
                swap_players(starters, bench, i, sub)
                sub['fatigue'] = max(0, sub['fatigue']-3)
                subs_made += 1
                if log: log.append(f"{team_name} - Sub for fouled out: {p['name']} -> {sub['name']}")
        elif p['fatigue'] > 25 or (p['fatigue']>18 and coach_style=='balanced'):
            eligible = [b for b in bench if not b.get('disqualified', False) and b.get('sit_until',0)<=game_seconds_left]
            if eligible:
                sub = min(eligible, key=lambda x:x['fatigue'])
                swap_players(starters, bench, i, sub)
                subs_made += 1
                if log: log.append(f"{team_name} - Sub for fatigue: {p['name']} -> {sub['name']}")
        elif coach_style=='conservative' and score_diff>10 and game_seconds_left>5*60:
            eligible = [b for b in bench if b.get('sit_until',0)<=game_seconds_left]
            if eligible:
                sub = min(eligible, key=lambda x:x['fatigue'])
                swap_players(starters, bench, i, sub)
                subs_made += 1
                if log: log.append(f"{team_name} - Sub to rest (leading): {p['name']} -> {sub['name']}")
    return subs_made

# -------------------------------
# Rebound logic
# -------------------------------
def get_rebound(off_court, def_court, is_offensive, log, team_name):
    if is_offensive:
        weights = [max(0.01, p['ORB'] - p['fatigue']*FATIGUE_REB_PENALTY) for p in off_court]
        rebounder = random.choices(off_court, weights=weights, k=1)[0]
        rebounder['stats']['off_reb'] += 1
        log.append(f"{team_name} - {rebounder['name']} grabbed offensive rebound")
    else:
        weights = [max(0.01, p['DRB'] - p['fatigue']*FATIGUE_REB_PENALTY) for p in def_court]
        rebounder = random.choices(def_court, weights=weights, k=1)[0]
        rebounder['stats']['def_reb'] += 1
        log.append(f"{team_name} - {rebounder['name']} grabbed defensive rebound")
    return rebounder

# -------------------------------
# Play type selection
# -------------------------------
def pick_play_type(team, opponent, is_transition, game_seconds_left, score_diff, coach_aggressive=False):
    probs = PLAYTYPE_BASE.copy()
    if is_transition:
        probs = {k: v*(1.5 if k=='transition' else 0.5) for k,v in probs.items()}
    if coach_aggressive and game_seconds_left<=COACH_AGGRESSIVE_MINUTES_LEFT*60 and score_diff<-COACH_AGGRESSIVE_DEFICIT:
        probs['transition']+=0.05; probs['iso']+=0.05; probs['spotup']+=0.03
    total=sum(probs.values())
    for k in probs: probs[k]/=total
    choices, weights = zip(*probs.items())
    return random.choices(choices, weights=weights, k=1)[0]

def shot_modifier_by_play(play_type):
    if play_type=='transition': return {'2P_mult':1.05,'3P_mult':0.9,'rebound_effort':0.9}
    if play_type=='pnr': return {'2P_mult':1.02,'3P_mult':1.03,'rebound_effort':1.0}
    if play_type=='iso': return {'2P_mult':0.95,'3P_mult':1.05,'rebound_effort':0.95}
    if play_type=='spotup': return {'2P_mult':0.95,'3P_mult':1.10,'rebound_effort':0.9}
    if play_type=='post': return {'2P_mult':1.10,'3P_mult':0.7,'rebound_effort':1.1}
    return {'2P_mult':1.0,'3P_mult':1.0,'rebound_effort':1.0}

# -------------------------------
# Simulate a single possession
# -------------------------------
def simulate_possession(off_court, def_court, game_seconds_left, team_fouls, is_home, log,
                        offense_team_name, defense_team_name, team_def_rating, coach_aggressive=False,
                        score_dict=None):
    """
    Simulate a single possession with detailed logging and running score.
    score_dict: dict with current score, updated each possession
    """
    is_transition = random.random() < 0.08
    shooter = random.choices(off_court, weights=[p['usage'] for p in off_court], k=1)[0]
    play_type = pick_play_type(off_court, def_court, is_transition, game_seconds_left,
                               score_diff=team_fouls.get('score_diff', 0), coach_aggressive=coach_aggressive)
    play_mod = shot_modifier_by_play(play_type)
    fatigue_penalty = min(0.25, shooter['fatigue'] * FATIGUE_PLAY_PENALTY)
    clutch_bonus = shooter.get('clutch_boost', 0.0) if game_seconds_left <= 120 else 0.0
    home_bonus = 0.02 if is_home else 0.0

    # Outcome weights
    base_two_weight = 0.50 * play_mod['2P_mult']
    base_three_weight = (0.20 + 0.5 * shooter['3P%']) * play_mod['3P_mult']
    if shooter['role'] == 'big': base_three_weight *= 0.5
    turnover_weight = shooter['TO']
    foul_weight = 0.14
    weights = [max(0.01, base_two_weight), max(0.01, base_three_weight),
               max(0.01, turnover_weight), max(0.01, foul_weight)]
    outcome = random.choices(['2P','3P','TO','Foul'], weights=weights, k=1)[0]

    possession_time = random.randint(6, 16)
    points = 0
    rebound_team = None
    play_log = ""

    if outcome in ['2P','3P']:
        base_prob = shooter['2P%'] if outcome == '2P' else shooter['3P%']
        effective_prob = base_prob * (play_mod['2P_mult'] if outcome == '2P' else play_mod['3P_mult']) \
                         - fatigue_penalty + clutch_bonus + home_bonus
        effective_prob = max(0.03, min(0.95, effective_prob))
        made = random.random() < effective_prob
        shooter['stats']['fga'] += 1
        if outcome == '3P': shooter['stats']['3pa'] += 1

        if made:
            pts = 2 if outcome == '2P' else 3
            shooter['stats']['fgm'] += 1
            if outcome == '3P': shooter['stats']['3pm'] += 1
            shooter['stats']['points'] += pts
            points += pts
            eligible_passers = [p for p in off_court if p['name'] != shooter['name']]
            assister_name = None
            if eligible_passers and random.random() < 0.28:
                assister = random.choices(eligible_passers, weights=[p['AST'] for p in eligible_passers], k=1)[0]
                assister['stats']['assists'] += 1
                assister_name = assister['name']
            play_log = f"{offense_team_name} - {shooter['name']} made {outcome}"
            if assister_name:
                play_log += f" (assisted by {assister_name})"
            play_log += f" [{play_type}]"
        else:
            off_reb_prob = 0.28 * play_mod['rebound_effort']
            is_offensive_reb = random.random() < off_reb_prob
            rebound_team = off_court if is_offensive_reb else def_court
            rebounder = get_rebound(off_court, def_court, is_offensive_reb, log, offense_team_name if is_offensive_reb else defense_team_name)
            possession_time += random.randint(2, 5)
            play_log = f"{offense_team_name} - {shooter['name']} missed {outcome} [{play_type}]; rebound by {rebounder['name']}"
    elif outcome == 'TO':
        rebound_team = def_court
        possession_time = random.randint(3, 8)
        shooter['stats']['turnovers'] += 1
        play_log = f"{offense_team_name} - {shooter['name']} committed a turnover"
    elif outcome == 'Foul':
        defender = random.choice(def_court)
        defender['fouls'] += 1
        defender['stats']['fouls'] = defender['fouls']
        shooter['stats']['fta'] += 1
        ft_made = int(random.random() < shooter['FT%'])
        shooter['stats']['ftm'] += ft_made
        shooter['stats']['points'] += ft_made
        points += ft_made
        rebound_team = off_court
        play_log = f"{defense_team_name} - {defender['name']} fouled {shooter['name']}; FT made: {ft_made}"

    # Update score
    if score_dict is not None:
        score_dict[offense_team_name] += points
        play_log += f" | Score: {score_dict[offense_team_name]} - {score_dict[defense_team_name]}"

    shooter['stats']['possessions'] += 1
    shooter['fatigue'] += 1 + (2 if play_type == 'transition' else 0)
    for p in def_court: p['fatigue'] += 0.2

    log.append(play_log)
    return points, possession_time, rebound_team, play_type


# -------------------------------
# Simulate a full game
# -------------------------------
def simulate_game(home_team_name, home_roster, away_team_name, away_roster, verbose=False):
    """
    Simulate a full possession-by-possession game with fatigue, substitutions,
    and running score log.
    """
    # Deep copy rosters so original data isn't modified
    home = copy.deepcopy(home_roster)
    away = copy.deepcopy(away_roster)
    home_starters, home_bench = home[:5], home[5:]
    away_starters, away_bench = away[:5], away[5:]

    # Initialize player stats and fatigue
    for p in home + away:
        p['fatigue'] = 0.0
        p['fouls'] = 0
        p['disqualified'] = False
        p['sit_until'] = 0
        for k in p['stats']:
            p['stats'][k] = 0

    # Game state
    score = {home_team_name: 0, away_team_name: 0}
    team_fouls = {home_team_name: 0, away_team_name: 0, 'score_diff': 0}
    possession_team = home_team_name
    time_left = GAME_SECONDS
    log = []
    possessions = 0
    team_possessions = {home_team_name: 0, away_team_name: 0}
    coach_state = {
        home_team_name: {'style': 'balanced', 'aggressive': False},
        away_team_name: {'style': 'balanced', 'aggressive': False}
    }

    # Main possession loop
    while time_left > 0:
        # Update aggressive coaching if behind late
        for tname in (home_team_name, away_team_name):
            other = away_team_name if tname == home_team_name else home_team_name
            lead = score[tname] - score[other]
            if lead < -COACH_AGGRESSIVE_DEFICIT and time_left <= COACH_AGGRESSIVE_MINUTES_LEFT*60:
                coach_state[tname]['aggressive'] = True
            else:
                coach_state[tname]['aggressive'] = False

        # Determine offense/defense
        if possession_team == home_team_name:
            off_court = home_starters
            def_court = away_starters
            is_home = True
            offense_name = home_team_name
            defense_name = away_team_name
            team_def_rating = 1.0
        else:
            off_court = away_starters
            def_court = home_starters
            is_home = False
            offense_name = away_team_name
            defense_name = home_team_name
            team_def_rating = 1.0

        # Update score differential for coach logic
        team_fouls['score_diff'] = score[offense_name] - score[defense_name]

        # Simulate possession
        pts, dur, rebound_team, play_type = simulate_possession(
            off_court, def_court, time_left, team_fouls, is_home, log,
            offense_name, defense_name, team_def_rating,
            coach_aggressive=coach_state[offense_name]['aggressive'],
            score_dict=score
        )

        score[possession_team] += pts
        team_possessions[possession_team] += 1
        possessions += 1
        time_left -= dur
        if time_left < 0: time_left = 0

        # Check for substitutions every few possessions
        if possessions % SUB_CHECK_EVERY_POSSESSIONS == 0:
            perform_subs(home_starters, home_bench, home_team_name, time_left,
                         score[home_team_name]-score[away_team_name], coach_style='balanced', log=log)
            perform_subs(away_starters, away_bench, away_team_name, time_left,
                         score[away_team_name]-score[home_team_name], coach_style='balanced', log=log)

        # Switch possession if defensive rebound or turnover
        if rebound_team is None or rebound_team == def_court:
            possession_team = away_team_name if possession_team == home_team_name else home_team_name

    # Return final game state
    return {
        'score': score,
        'players': {p['name']: p['stats'] for p in home + away},
        'log': log,
        'team_possessions': team_possessions
    }




# -------------------------------
# Generate top performers summary with advanced stats
# -------------------------------
def summarize_top_performers_advanced(avg_player_stats, team_totals, top_n=5):
    """
    Summarize top performers across multiple simulations including advanced metrics:
    - Usage %
    - Offensive/Defensive Rebound %
    - Assist %
    - Steal %, Block %, Turnover %
    Returns a sorted list of dicts by efficiency.
    
    team_totals: dict with keys 'possessions', 'team_fg', 'team_fga', etc.
    """
    summary = []

    for pname, stats in avg_player_stats.items():
        pts = stats.get('points', 0)
        off_reb = stats.get('off_reb', 0)
        def_reb = stats.get('def_reb', 0)
        ast = stats.get('assists', 0)
        stl = stats.get('steals', 0)
        blk = stats.get('blocks', 0)
        tov = stats.get('turnovers', 0)
        fgm = stats.get('fgm', 0)
        fga = stats.get('fga', 1)
        ftm = stats.get('ftm', 0)
        fta = stats.get('fta', 1)
        poss = stats.get('possessions', 1)

        # Efficiency
        eff = pts + off_reb + def_reb + ast + stl + blk - tov

        # Advanced stats calculations
        usage = (poss / team_totals.get('possessions', 1)) * 100
        off_reb_pct = (off_reb / max(1, team_totals.get('team_off_reb', 1))) * 100
        def_reb_pct = (def_reb / max(1, team_totals.get('team_def_reb', 1))) * 100
        ast_pct = (ast / max(1, team_totals.get('team_fg', 1))) * 100
        stl_pct = (stl / max(1, team_totals.get('team_possessions', 1))) * 100
        blk_pct = (blk / max(1, team_totals.get('team_opp_fga', 1))) * 100
        tov_pct = (tov / max(1, poss)) * 100

        summary.append({
            'player': pname,
            'points': round(pts,1),
            'off_reb': round(off_reb,1),
            'def_reb': round(def_reb,1),
            'assists': round(ast,1),
            'steals': round(stl,1),
            'blocks': round(blk,1),
            'turnovers': round(tov,1),
            'efficiency': round(eff,1),
            'fg_pct': round(fgm/fga if fga>0 else 0, 2),
            'ft_pct': round(ftm/fta if fta>0 else 0, 2),
            'usage_pct': round(usage,1),
            'off_reb_pct': round(off_reb_pct,1),
            'def_reb_pct': round(def_reb_pct,1),
            'ast_pct': round(ast_pct,1),
            'stl_pct': round(stl_pct,1),
            'blk_pct': round(blk_pct,1),
            'tov_pct': round(tov_pct,1)
        })

    # Sort by efficiency descending
    summary_sorted = sorted(summary, key=lambda x: x['efficiency'], reverse=True)

    return summary_sorted[:top_n]


def run_simulation(home_team_name, away_team_name, home_roster, away_roster, n_games=1, verbose=False):
    """
    Run either a single game or a Monte Carlo simulation of multiple games.
    
    Args:
        home_team_name (str): Name of the home team.
        away_team_name (str): Name of the away team.
        home_roster (list): List of home team player dicts.
        away_roster (list): List of away team player dicts.
        n_games (int): Number of games to simulate. 1 = single game.
        verbose (bool): If True, prints each game score.
        
    Returns:
        dict or list: Single game dict if n_games=1, else list of game dicts with average summary.
    """
    if n_games == 1:
        # Single game
        game_result = simulate_game(home_team_name, home_roster, away_team_name, away_roster, verbose=verbose)
        if verbose:
            print(f"Final Score: {home_team_name} {game_result['score'][home_team_name]} - {away_team_name} {game_result['score'][away_team_name]}")
        return game_result
    else:
        # Multi-game Monte Carlo
        results = []
        for i in range(n_games):
            result = simulate_game(home_team_name, home_roster, away_team_name, away_roster)
            results.append(result)
            if verbose:
                print(f"Game {i+1}: {home_team_name} {result['score'][home_team_name]} - {away_team_name} {result['score'][away_team_name]}")
        
        # Compute averages
        avg_home = sum(r['score'][home_team_name] for r in results) / n_games
        avg_away = sum(r['score'][away_team_name] for r in results) / n_games
        print(f"\nAverage over {n_games} games: {home_team_name} {avg_home:.1f} - {away_team_name} {avg_away:.1f}")
        return results





# Build dummy rosters
home_roster = [create_player(f"LAL_{r}", role=random.choice(['pg','wing','big'])) for r in
               ["PG","W1","W2","B1","B2","BEN1","BEN2","BEN3","BEN4","BEN5"]]
away_roster = [create_player(f"BOS_{r}", role=random.choice(['pg','wing','big'])) for r in
               ["PG","W1","W2","B1","B2","BEN1","BEN2","BEN3","BEN4","BEN5"]]

# Run single game (with detailed possession log)
single_game = run_simulation("LAL", "BOS", home_roster, away_roster, n_games=1, verbose=True)

# Run 20-game Monte Carlo simulation (no log saved)
multi_game = run_simulation("LAL", "BOS", home_roster, away_roster, n_games=20, verbose=True)


Final Score: LAL 98 - BOS 88
Game 1: LAL 98 - BOS 50
Game 2: LAL 68 - BOS 78
Game 3: LAL 72 - BOS 70
Game 4: LAL 102 - BOS 64
Game 5: LAL 70 - BOS 106
Game 6: LAL 78 - BOS 110
Game 7: LAL 60 - BOS 102
Game 8: LAL 92 - BOS 110
Game 9: LAL 98 - BOS 74
Game 10: LAL 124 - BOS 54
Game 11: LAL 102 - BOS 102
Game 12: LAL 50 - BOS 102
Game 13: LAL 114 - BOS 84
Game 14: LAL 74 - BOS 94
Game 15: LAL 94 - BOS 74
Game 16: LAL 80 - BOS 62
Game 17: LAL 40 - BOS 80
Game 18: LAL 78 - BOS 64
Game 19: LAL 98 - BOS 114
Game 20: LAL 126 - BOS 98

Average over 20 games: LAL 85.9 - BOS 84.6


In [14]:
import random
import copy
from statistics import mean

# -------------------------------
# Config / Tunables
# -------------------------------
GAME_MINUTES = 48
SECONDS_PER_MIN = 60
GAME_SECONDS = GAME_MINUTES * SECONDS_PER_MIN
SUB_CHECK_EVERY_POSSESSIONS = 4
FATIGUE_PLAY_PENALTY = 0.006
FATIGUE_REB_PENALTY = 0.03
COACH_AGGRESSIVE_DEFICIT = 8
COACH_AGGRESSIVE_MINUTES_LEFT = 6

PLAYTYPE_BASE = {
    'transition': 0.10,
    'pnr': 0.30,
    'iso': 0.20,
    'spotup': 0.25,
    'post': 0.15
}

# -------------------------------
# Player creation
# -------------------------------
def create_player(name, role='wing', mpg=30, stats_input=None):
    if stats_input is None: stats_input = {}
    return {
        'name': name,
        'role': role,
        'mpg': mpg,
        '2PA': stats_input.get('2PA', random.uniform(5,15)),
        '2P%': stats_input.get('2P%', random.uniform(0.45,0.55)),
        '3PA': stats_input.get('3PA', random.uniform(0,6)),
        '3P%': stats_input.get('3P%', random.uniform(0.33,0.38)),
        'FTA': stats_input.get('FTA', random.uniform(1,5)),
        'FT%': stats_input.get('FT%', random.uniform(0.7,0.9)),
        'ORB': stats_input.get('ORB', random.uniform(0.5,3)),
        'DRB': stats_input.get('DRB', random.uniform(1.5,8)),
        'AST': stats_input.get('AST', random.uniform(1,8)),
        'STL': stats_input.get('STL', random.uniform(0,2)),
        'BLK': stats_input.get('BLK', random.uniform(0,2)),
        'TO': stats_input.get('TO', random.uniform(1,4)),
        'PF': stats_input.get('PF', random.uniform(1,4)),
        'usage': stats_input.get('usage', random.uniform(15,30)),
        'fatigue': 0.0,
        'fouls': 0,
        'disqualified': False,
        'sit_until': 0,
        'stats': {k:0 for k in ['points','fouls','possessions','assists','off_reb','def_reb','steals','blocks','turnovers','fgm','fga','3pm','3pa','ftm','fta']}
    }

# -------------------------------
# Helper functions
# -------------------------------
def swap_players(on_court, bench, idx_on, bench_player):
    player_out = on_court[idx_on]
    if bench_player not in bench: return False
    bench.remove(bench_player)
    bench.append(player_out)
    on_court[idx_on] = bench_player
    return True

def perform_subs(starters, bench, team_name, game_seconds_left, score_diff, coach_style='balanced', log=None):
    subs_made = 0
    for i, p in enumerate(list(starters)):
        if p.get('disqualified', False) or p['fouls'] >= 6:
            eligible = [b for b in bench if not b.get('disqualified', False) and b.get('sit_until',0)<=game_seconds_left]
            if eligible:
                sub = max(eligible, key=lambda x: x['usage'])
                swap_players(starters, bench, i, sub)
                sub['fatigue'] = max(0, sub['fatigue']-3)
                subs_made += 1
                if log: log.append(f"{team_name} - Sub for fouled out: {p['name']} -> {sub['name']}")
        elif p['fatigue'] > 25 or (p['fatigue']>18 and coach_style=='balanced'):
            eligible = [b for b in bench if not b.get('disqualified', False) and b.get('sit_until',0)<=game_seconds_left]
            if eligible:
                sub = min(eligible, key=lambda x:x['fatigue'])
                swap_players(starters, bench, i, sub)
                subs_made += 1
                if log: log.append(f"{team_name} - Sub for fatigue: {p['name']} -> {sub['name']}")
    return subs_made

def get_rebound(off_court, def_court, is_offensive, log, team_name):
    weights = [max(0.01, (p['ORB'] if is_offensive else p['DRB']) - p['fatigue']*FATIGUE_REB_PENALTY) for p in (off_court if is_offensive else def_court)]
    rebounder = random.choices(off_court if is_offensive else def_court, weights=weights, k=1)[0]
    key = 'off_reb' if is_offensive else 'def_reb'
    rebounder['stats'][key] += 1
    log.append(f"{team_name} - {rebounder['name']} grabbed {'offensive' if is_offensive else 'defensive'} rebound")
    return rebounder

def pick_play_type(team, opponent, is_transition, game_seconds_left, score_diff, coach_aggressive=False):
    probs = PLAYTYPE_BASE.copy()
    if is_transition:
        probs = {k: v*(1.5 if k=='transition' else 0.5) for k,v in probs.items()}
    if coach_aggressive and game_seconds_left<=COACH_AGGRESSIVE_MINUTES_LEFT*60 and score_diff<-COACH_AGGRESSIVE_DEFICIT:
        probs['transition'] += 0.05; probs['iso'] += 0.05; probs['spotup'] += 0.03
    total=sum(probs.values())
    for k in probs: probs[k]/=total
    return random.choices(list(probs.keys()), weights=list(probs.values()), k=1)[0]

def shot_modifier_by_play(play_type):
    return {
        'transition': {'2P_mult':1.05,'3P_mult':0.9,'rebound_effort':0.9},
        'pnr': {'2P_mult':1.02,'3P_mult':1.03,'rebound_effort':1.0},
        'iso': {'2P_mult':0.95,'3P_mult':1.05,'rebound_effort':0.95},
        'spotup': {'2P_mult':0.95,'3P_mult':1.10,'rebound_effort':0.9},
        'post': {'2P_mult':1.10,'3P_mult':0.7,'rebound_effort':1.1}
    }.get(play_type, {'2P_mult':1.0,'3P_mult':1.0,'rebound_effort':1.0})

# -------------------------------
# Simulate a possession
# -------------------------------
def simulate_possession(off_court, def_court, game_seconds_left, team_fouls, is_home, log,
                        offense_team_name, defense_team_name, team_def_rating, coach_aggressive=False,
                        score_dict=None):
    is_transition = random.random() < 0.08
    shooter = random.choices(off_court, weights=[p['usage'] for p in off_court], k=1)[0]
    play_type = pick_play_type(off_court, def_court, is_transition, game_seconds_left, score_diff=team_fouls.get('score_diff',0), coach_aggressive=coach_aggressive)
    play_mod = shot_modifier_by_play(play_type)
    fatigue_penalty = min(0.25, shooter['fatigue']*FATIGUE_PLAY_PENALTY)
    home_bonus = 0.02 if is_home else 0.0

    base_two_weight = 0.50*play_mod['2P_mult']
    base_three_weight = (0.20+0.5*shooter['3P%'])*play_mod['3P_mult']
    if shooter['role']=='big': base_three_weight *= 0.5
    turnover_weight = shooter['TO']
    foul_weight = 0.14
    weights = [max(0.01,base_two_weight), max(0.01,base_three_weight), max(0.01,turnover_weight), max(0.01,foul_weight)]
    outcome = random.choices(['2P','3P','TO','Foul'], weights=weights,k=1)[0]

    possession_time = random.randint(6,16)
    points = 0
    rebound_team = None

    if outcome in ['2P','3P']:
        base_prob = shooter['2P%'] if outcome=='2P' else shooter['3P%']
        effective_prob = max(0.03, min(0.95, base_prob*(play_mod['2P_mult'] if outcome=='2P' else play_mod['3P_mult'])-fatigue_penalty+home_bonus))
        made = random.random() < effective_prob
        shooter['stats']['fga'] += 1
        if outcome=='3P': shooter['stats']['3pa'] +=1
        if made:
            pts = 2 if outcome=='2P' else 3
            shooter['stats']['fgm'] += 1
            if outcome=='3P': shooter['stats']['3pm'] += 1
            shooter['stats']['points'] += pts
            points += pts
            eligible_passers = [p for p in off_court if p['name']!=shooter['name']]
            if eligible_passers and random.random()<0.28:
                assister = random.choices(eligible_passers, weights=[p['AST'] for p in eligible_passers],k=1)[0]
                assister['stats']['assists'] +=1
        else:
            off_reb_prob = 0.28*play_mod['rebound_effort']
            is_offensive_reb = random.random() < off_reb_prob
            rebound_team = off_court if is_offensive_reb else def_court
            get_rebound(off_court, def_court, is_offensive_reb, log, offense_team_name if is_offensive_reb else defense_team_name)
            possession_time += random.randint(2,5)
    elif outcome=='TO':
        rebound_team = def_court
        possession_time = random.randint(3,8)
        shooter['stats']['turnovers'] +=1
    elif outcome=='Foul':
        defender=random.choice(def_court)
        defender['fouls']+=1
        defender['stats']['fouls']=defender['fouls']
        shooter['stats']['fta']+=1
        ft_made = int(random.random() < shooter['FT%'])
        shooter['stats']['ftm'] += ft_made
        shooter['stats']['points'] += ft_made
        points += ft_made
        rebound_team = off_court

    shooter['stats']['possessions'] +=1
    shooter['fatigue'] += 1+(2 if play_type=='transition' else 0)
    for p in def_court: p['fatigue'] += 0.2

    if score_dict is not None:
        score_dict[offense_team_name] += points

    log.append(f"{offense_team_name} - {shooter['name']} {outcome}, Score: {score_dict[offense_team_name]}-{score_dict[defense_team_name]}" if score_dict else f"{offense_team_name} - {shooter['name']} {outcome}")
    return points, possession_time, rebound_team, play_type

# -------------------------------
# Simulate full game
# -------------------------------
def simulate_game(home_team_name, home_roster, away_team_name, away_roster, verbose=False):
    home = copy.deepcopy(home_roster)
    away = copy.deepcopy(away_roster)
    home_starters, home_bench = home[:5], home[5:]
    away_starters, away_bench = away[:5], away[5:]

    for p in home+away:
        p['fatigue']=0; p['fouls']=0; p['disqualified']=False; p['sit_until']=0
        for k in p['stats']: p['stats'][k]=0

    score = {home_team_name:0, away_team_name:0}
    team_fouls = {home_team_name:0, away_team_name:0,'score_diff':0}
    possession_team = home_team_name
    time_left = GAME_SECONDS
    log=[]
    possessions=0
    team_possessions={home_team_name:0, away_team_name:0}
    coach_state={home_team_name:{'style':'balanced','aggressive':False}, away_team_name:{'style':'balanced','aggressive':False}}

    while time_left>0:
        for tname in (home_team_name, away_team_name):
            other = away_team_name if tname==home_team_name else home_team_name
            lead = score[tname]-score[other]
            coach_state[tname]['aggressive'] = lead < -COACH_AGGRESSIVE_DEFICIT and time_left <= COACH_AGGRESSIVE_MINUTES_LEFT*60

        if possession_team==home_team_name:
            off_court=home_starters; def_court=away_starters; is_home=True; offense_name=home_team_name; defense_name=away_team_name
        else:
            off_court=away_starters; def_court=home_starters; is_home=False; offense_name=away_team_name; defense_name=home_team_name

        team_fouls['score_diff']=score[offense_name]-score[defense_name]

        pts, dur, rebound_team, play_type = simulate_possession(
            off_court, def_court, time_left, team_fouls, is_home, log,
            offense_name, defense_name, 1.0, coach_aggressive=coach_state[offense_name]['aggressive'],
            score_dict=score
        )

        team_possessions[possession_team]+=1
        possessions+=1
        time_left -= dur
        if time_left<0: time_left=0

        if possessions % SUB_CHECK_EVERY_POSSESSIONS==0:
            perform_subs(home_starters, home_bench, home_team_name, time_left, score[home_team_name]-score[away_team_name], log=log)
            perform_subs(away_starters, away_bench, away_team_name, time_left, score[away_team_name]-score[home_team_name], log=log)

        if rebound_team is None or rebound_team==def_court:
            possession_team = away_team_name if possession_team==home_team_name else home_team_name

    return {'score':score,'players':{p['name']:p['stats'] for p in home+away},'log':log,'team_possessions':team_possessions}

# -------------------------------
# Summarize top performers
# -------------------------------
def summarize_top_performers_advanced(avg_player_stats, team_totals, top_n=5):
    summary=[]
    for pname, stats in avg_player_stats.items():
        pts = stats.get('points',0); off_reb=stats.get('off_reb',0); def_reb=stats.get('def_reb',0)
        ast=stats.get('assists',0); stl=stats.get('steals',0); blk=stats.get('blocks',0)
        tov=stats.get('turnovers',0); fgm=stats.get('fgm',0); fga=stats.get('fga',1)
        ftm=stats.get('ftm',0); fta=stats.get('fta',1); poss=stats.get('possessions',1)
        eff = pts + off_reb + def_reb + ast + stl + blk - tov
        usage = (poss / max(1, team_totals.get('possessions',1)))*100
        off_reb_pct = (off_reb / max(1, team_totals.get('team_off_reb',1)))*100
        def_reb_pct = (def_reb / max(1, team_totals.get('team_def_reb',1)))*100
        ast_pct = (ast / max(1, team_totals.get('team_fg',1)))*100
        stl_pct = (stl / max(1, team_totals.get('possessions',1)))*100
        blk_pct = (blk / max(1, team_totals.get('team_fg',1)))*100
        tov_pct = (tov / max(1, poss))*100
        summary.append({'player':pname,'points':round(pts,1),'off_reb':round(off_reb,1),'def_reb':round(def_reb,1),
                        'assists':round(ast,1),'steals':round(stl,1),'blocks':round(blk,1),'turnovers':round(tov,1),
                        'efficiency':round(eff,1),'fg_pct':round(fgm/fga if fga>0 else 0,2),
                        'ft_pct':round(ftm/fta if fta>0 else 0,2),'usage_pct':round(usage,1),
                        'off_reb_pct':round(off_reb_pct,1),'def_reb_pct':round(def_reb_pct,1),
                        'ast_pct':round(ast_pct,1),'stl_pct':round(stl_pct,1),'blk_pct':round(blk_pct,1),
                        'tov_pct':round(tov_pct,1)})
    return sorted(summary,key=lambda x:x['efficiency'],reverse=True)[:top_n]

# -------------------------------
# Run Simulation (Single or Multi)
# -------------------------------
def run_simulation(home_team_name, away_team_name, home_roster, away_roster, n_games=1, verbose=False, return_avg_player_stats=False):
    if n_games==1:
        game_result = simulate_game(home_team_name, home_roster, away_team_name, away_roster, verbose=verbose)
        if verbose: print(f"Final Score: {home_team_name} {game_result['score'][home_team_name]} - {away_team_name} {game_result['score'][away_team_name]}")
        return game_result

    # Multi-game Monte Carlo
    results=[]
    cumulative_player_stats={}
    cumulative_team_stats={home_team_name:{'possessions':0,'team_fg':0,'team_fga':0,'team_off_reb':0,'team_def_reb':0,'team_fta':0,'team_ftm':0},
                           away_team_name:{'possessions':0,'team_fg':0,'team_fga':0,'team_off_reb':0,'team_def_reb':0,'team_fta':0,'team_ftm':0}}
    for i in range(n_games):
        result=simulate_game(home_team_name, home_roster, away_team_name, away_roster)
        results.append(result)
        if verbose: print(f"Game {i+1}: {home_team_name} {result['score'][home_team_name]} - {away_team_name} {result['score'][away_team_name]}")
        # accumulate player stats
        for pname, stats in result['players'].items():
            if pname not in cumulative_player_stats: cumulative_player_stats[pname]={k:0 for k in stats}
            for k,v in stats.items(): cumulative_player_stats[pname][k]+=v
        # accumulate team stats
        for tname in [home_team_name, away_team_name]:
            cumulative_team_stats[tname]['possessions']+=result['team_possessions'][tname]
            team_roster = home_roster if tname==home_team_name else away_roster
            cumulative_team_stats[tname]['team_fg'] += sum(p['stats']['fgm'] for p in team_roster)
            cumulative_team_stats[tname]['team_fga'] += sum(p['stats']['fga'] for p in team_roster)
            cumulative_team_stats[tname]['team_off_reb'] += sum(p['stats']['off_reb'] for p in team_roster)
            cumulative_team_stats[tname]['team_def_reb'] += sum(p['stats']['def_reb'] for p in team_roster)
            cumulative_team_stats[tname]['team_fta'] += sum(p['stats']['fta'] for p in team_roster)
            cumulative_team_stats[tname]['team_ftm'] += sum(p['stats']['ftm'] for p in team_roster)

    # averages
    for pname, stats in cumulative_player_stats.items():
        for k in stats: stats[k] /= n_games
    for tname in cumulative_team_stats:
        for k in cumulative_team_stats[tname]: cumulative_team_stats[tname][k]/=n_games

    avg_home_score = sum(r['score'][home_team_name] for r in results)/n_games
    avg_away_score = sum(r['score'][away_team_name] for r in results)/n_games
    print(f"\nAverage Score over {n_games} games: {home_team_name} {avg_home_score:.1f} - {away_team_name} {avg_away_score:.1f}")

    if return_avg_player_stats:
        top_players_avg = summarize_top_performers_advanced(cumulative_player_stats, cumulative_team_stats[home_team_name], top_n=5)
        return results, top_players_avg, cumulative_team_stats
    return results

In [15]:
# Build dummy rosters
home_roster = [create_player(f"LAL_{r}", role=random.choice(['pg','wing','big'])) for r in ["PG","W1","W2","B1","B2","BEN1","BEN2","BEN3","BEN4","BEN5"]]
away_roster = [create_player(f"BOS_{r}", role=random.choice(['pg','wing','big'])) for r in ["PG","W1","W2","B1","B2","BEN1","BEN2","BEN3","BEN4","BEN5"]]

# Single game detailed log
single_game = run_simulation("LAL","BOS",home_roster,away_roster,n_games=1,verbose=True)

# 20-game Monte Carlo
multi_game_results, top_players_avg, avg_team_stats = run_simulation("LAL","BOS",home_roster,away_roster,n_games=20,verbose=True,return_avg_player_stats=True)

# Top 5 averaged players
print("\n--- Top 5 Players Over 20 Games ---")
for p in top_players_avg:
    print(f"{p['player']}: Points {p['points']}, Efficiency {p['efficiency']}, Usage {p['usage_pct']}%")

# Team-level stats
print("\n--- Average Team Stats Over 20 Games ---")
for tname, stats in avg_team_stats.items():
    fg_pct = stats['team_fg']/stats['team_fga'] if stats['team_fga']>0 else 0
    ft_pct = stats['team_ftm']/stats['team_fta'] if stats['team_fta']>0 else 0
    print(f"{tname}: FG% {fg_pct:.3f}, FT% {ft_pct:.3f}, OffReb {stats['team_off_reb']:.1f}, DefReb {stats['team_def_reb']:.1f}, Possessions {stats['possessions']:.1f}")


Final Score: LAL 50 - BOS 43
Game 1: LAL 46 - BOS 52
Game 2: LAL 55 - BOS 32
Game 3: LAL 42 - BOS 44
Game 4: LAL 49 - BOS 45
Game 5: LAL 34 - BOS 43
Game 6: LAL 35 - BOS 34
Game 7: LAL 47 - BOS 67
Game 8: LAL 47 - BOS 38
Game 9: LAL 44 - BOS 51
Game 10: LAL 41 - BOS 42
Game 11: LAL 78 - BOS 45
Game 12: LAL 54 - BOS 28
Game 13: LAL 48 - BOS 52
Game 14: LAL 51 - BOS 52
Game 15: LAL 30 - BOS 48
Game 16: LAL 35 - BOS 31
Game 17: LAL 47 - BOS 41
Game 18: LAL 36 - BOS 46
Game 19: LAL 54 - BOS 32
Game 20: LAL 60 - BOS 37

Average Score over 20 games: LAL 46.6 - BOS 43.0

--- Top 5 Players Over 20 Games ---
LAL_BEN3: Points 7.5, Efficiency 3.7, Usage 9.1%
BOS_BEN2: Points 5.3, Efficiency 1.8, Usage 9.1%
LAL_BEN4: Points 7.0, Efficiency -0.3, Usage 10.9%
LAL_W1: Points 6.5, Efficiency -0.6, Usage 9.1%
BOS_BEN1: Points 5.5, Efficiency -1.7, Usage 9.6%

--- Average Team Stats Over 20 Games ---
LAL: FG% 0.000, FT% 0.000, OffReb 0.0, DefReb 0.0, Possessions 185.6
BOS: FG% 0.000, FT% 0.000, OffReb 0

In [18]:
import random
import copy
from statistics import mean

# -------------------------------
# Config / Tunables
# -------------------------------
GAME_MINUTES = 48
SECONDS_PER_MIN = 60
GAME_SECONDS = GAME_MINUTES * SECONDS_PER_MIN
QUARTER_SECONDS = 12 * SECONDS_PER_MIN
SUB_CHECK_EVERY_POSSESSIONS = 4
FATIGUE_PLAY_PENALTY = 0.006
FATIGUE_REB_PENALTY = 0.03
FATIGUE_DEF_PENALTY = 0.004
COACH_AGGRESSIVE_DEFICIT = 8
COACH_AGGRESSIVE_MINUTES_LEFT = 6

PLAYTYPE_BASE = {
    'transition': 0.10,
    'pnr': 0.30,
    'iso': 0.20,
    'spotup': 0.25,
    'post': 0.15
}

# -------------------------------
# Player creation
# -------------------------------
def create_player(name, role='wing', mpg=30, stats_input=None):
    if stats_input is None: stats_input={}
    return {
        'name': name,
        'role': role,
        'mpg': mpg,
        '2PA': stats_input.get('2PA', random.uniform(5,15)),
        '2P%': stats_input.get('2P%', random.uniform(0.45,0.55)),
        '3PA': stats_input.get('3PA', random.uniform(0,6)),
        '3P%': stats_input.get('3P%', random.uniform(0.33,0.38)),
        'FTA': stats_input.get('FTA', random.uniform(1,5)),
        'FT%': stats_input.get('FT%', random.uniform(0.7,0.9)),
        'ORB': stats_input.get('ORB', random.uniform(0.5,3)),
        'DRB': stats_input.get('DRB', random.uniform(1.5,8)),
        'AST': stats_input.get('AST', random.uniform(1,8)),
        'STL': stats_input.get('STL', random.uniform(0,2)),
        'BLK': stats_input.get('BLK', random.uniform(0,2)),
        'TO': stats_input.get('TO', random.uniform(1,4)),
        'PF': stats_input.get('PF', random.uniform(1,4)),
        'usage': stats_input.get('usage', random.uniform(15,30)),
        'fatigue': 0.0,
        'fouls': 0,
        'disqualified': False,
        'sit_until': 0,
        'stats': {
            'points':0,'fouls':0,'possessions':0,'assists':0,'off_reb':0,'def_reb':0,
            'steals':0,'blocks':0,'turnovers':0,'fgm':0,'fga':0,'3pm':0,'3pa':0,'ftm':0,'fta':0
        }
    }

# -------------------------------
# Helper: swap players
# -------------------------------
def swap_players(on_court, bench, idx_on, bench_player):
    player_out = on_court[idx_on]
    if bench_player not in bench: return False
    bench.remove(bench_player)
    bench.append(player_out)
    on_court[idx_on] = bench_player
    return True

# -------------------------------
# Substitution logic
# -------------------------------
def perform_subs(starters, bench, team_name, game_seconds_left, score_diff, coach_style='balanced', log=None):
    subs_made = 0
    for i, p in enumerate(list(starters)):
        if p.get('disqualified', False) or p['fouls'] >= 6:
            eligible = [b for b in bench if not b.get('disqualified', False) and b.get('sit_until',0)<=game_seconds_left]
            if eligible:
                sub = max(eligible, key=lambda x: x['usage'])
                swap_players(starters, bench, i, sub)
                sub['fatigue'] = max(0, sub['fatigue']-3)
                subs_made += 1
                if log: log.append(f"{team_name} - Sub for fouled out: {p['name']} -> {sub['name']}")
        elif p['fatigue'] > 25 or (p['fatigue']>18 and coach_style=='balanced'):
            eligible = [b for b in bench if not b.get('disqualified', False) and b.get('sit_until',0)<=game_seconds_left]
            if eligible:
                sub = min(eligible, key=lambda x:x['fatigue'])
                swap_players(starters, bench, i, sub)
                subs_made += 1
                if log: log.append(f"{team_name} - Sub for fatigue: {p['name']} -> {sub['name']}")
        elif coach_style=='conservative' and score_diff>10 and game_seconds_left>5*60:
            eligible = [b for b in bench if b.get('sit_until',0)<=game_seconds_left]
            if eligible:
                sub = min(eligible, key=lambda x:x['fatigue'])
                swap_players(starters, bench, i, sub)
                subs_made += 1
                if log: log.append(f"{team_name} - Sub to rest (leading): {p['name']} -> {sub['name']}")
    return subs_made

# -------------------------------
# Rebound logic
# -------------------------------
def get_rebound(off_court, def_court, is_offensive, log, team_name):
    if is_offensive:
        weights = [max(0.01, p['ORB'] - p['fatigue']*FATIGUE_REB_PENALTY) for p in off_court]
        rebounder = random.choices(off_court, weights=weights, k=1)[0]
        rebounder['stats']['off_reb'] += 1
        log.append(f"{team_name} - {rebounder['name']} grabbed offensive rebound")
    else:
        weights = [max(0.01, p['DRB'] - p['fatigue']*FATIGUE_REB_PENALTY) for p in def_court]
        rebounder = random.choices(def_court, weights=weights, k=1)[0]
        rebounder['stats']['def_reb'] += 1
        log.append(f"{team_name} - {rebounder['name']} grabbed defensive rebound")
    return rebounder

# -------------------------------
# Play type selection
# -------------------------------
def pick_play_type(team, opponent, is_transition, game_seconds_left, score_diff, coach_aggressive=False):
    probs = PLAYTYPE_BASE.copy()
    if is_transition:
        probs = {k: v*(1.5 if k=='transition' else 0.5) for k,v in probs.items()}
    if coach_aggressive and game_seconds_left<=COACH_AGGRESSIVE_MINUTES_LEFT*60 and score_diff<-COACH_AGGRESSIVE_DEFICIT:
        probs['transition']+=0.05; probs['iso']+=0.05; probs['spotup']+=0.03
    total=sum(probs.values())
    for k in probs: probs[k]/=total
    choices, weights = zip(*probs.items())
    return random.choices(choices, weights=weights, k=1)[0]

def shot_modifier_by_play(play_type):
    if play_type=='transition': return {'2P_mult':1.05,'3P_mult':0.9,'rebound_effort':0.9}
    if play_type=='pnr': return {'2P_mult':1.02,'3P_mult':1.03,'rebound_effort':1.0}
    if play_type=='iso': return {'2P_mult':0.95,'3P_mult':1.05,'rebound_effort':0.95}
    if play_type=='spotup': return {'2P_mult':0.95,'3P_mult':1.10,'rebound_effort':0.9}
    if play_type=='post': return {'2P_mult':1.10,'3P_mult':0.7,'rebound_effort':1.1}
    return {'2P_mult':1.0,'3P_mult':1.0,'rebound_effort':1.0}

# -------------------------------
# Single possession simulation
# -------------------------------
def simulate_possession(off_court, def_court, game_seconds_left, team_fouls, is_home, log,
                        offense_team_name, defense_team_name, team_def_rating, coach_aggressive=False,
                        score_dict=None):
    is_transition = random.random() < 0.08
    shooter = random.choices(off_court, weights=[p['usage'] for p in off_court], k=1)[0]
    play_type = pick_play_type(off_court, def_court, is_transition, game_seconds_left,
                               score_diff=team_fouls.get('score_diff', 0), coach_aggressive=coach_aggressive)
    play_mod = shot_modifier_by_play(play_type)
    fatigue_penalty = min(0.25, shooter['fatigue'] * FATIGUE_PLAY_PENALTY)
    clutch_bonus = shooter.get('clutch_boost', 0.0) if game_seconds_left <= 120 else 0.0
    home_bonus = 0.02 if is_home else 0.0

    base_two_weight = 0.50 * play_mod['2P_mult']
    base_three_weight = (0.20 + 0.5 * shooter['3P%']) * play_mod['3P_mult']
    if shooter['role']=='big': base_three_weight *= 0.5
    turnover_weight = shooter['TO']
    foul_weight = 0.14
    weights = [max(0.01, base_two_weight), max(0.01, base_three_weight), max(0.01, turnover_weight), max(0.01, foul_weight)]
    outcome = random.choices(['2P','3P','TO','Foul'], weights=weights, k=1)[0]

    possession_time = random.randint(6,16)
    points=0
    rebound_team=None
    play_log=""

    if outcome in ['2P','3P']:
        base_prob = shooter['2P%'] if outcome=='2P' else shooter['3P%']
        effective_prob = base_prob*(play_mod['2P_mult'] if outcome=='2P' else play_mod['3P_mult']) - fatigue_penalty + clutch_bonus + home_bonus
        effective_prob = max(0.03, min(0.95, effective_prob))
        made = random.random() < effective_prob
        shooter['stats']['fga'] += 1
        if outcome=='3P': shooter['stats']['3pa'] += 1
        if made:
            pts = 2 if outcome=='2P' else 3
            shooter['stats']['fgm'] += 1
            if outcome=='3P': shooter['stats']['3pm'] += 1
            shooter['stats']['points'] += pts
            points += pts
            eligible_passers = [p for p in off_court if p['name']!=shooter['name']]
            if eligible_passers and random.random()<0.28:
                assister=random.choices(eligible_passers, weights=[p['AST'] for p in eligible_passers],k=1)[0]
                assister['stats']['assists'] += 1
            play_log = f"{offense_team_name} - {shooter['name']} made {outcome} [{play_type}]"
        else:
            off_reb_prob = 0.28*play_mod['rebound_effort']
            is_offensive_reb = random.random()<off_reb_prob
            rebound_team = off_court if is_offensive_reb else def_court
            rebounder = get_rebound(off_court, def_court, is_offensive_reb, log, offense_team_name if is_offensive_reb else defense_team_name)
            possession_time += random.randint(2,5)
            play_log = f"{offense_team_name} - {shooter['name']} missed {outcome} [{play_type}]; rebound by {rebounder['name']}"
    elif outcome=='TO':
        rebound_team=def_court
        possession_time=random.randint(3,8)
        shooter['stats']['turnovers'] += 1
        play_log = f"{offense_team_name} - {shooter['name']} committed TO"
    elif outcome=='Foul':
        defender=random.choice(def_court)
        defender['fouls'] += 1
        defender['stats']['fouls'] = defender['fouls']
        shooter['stats']['fta'] += 1
        ft_made = int(random.random() < shooter['FT%'])
        shooter['stats']['ftm'] += ft_made
        shooter['stats']['points'] += ft_made
        points += ft_made
        rebound_team = off_court
        play_log = f"{defense_team_name} - {defender['name']} fouled {shooter['name']}; FT made: {ft_made}"

    if score_dict is not None:
        score_dict[offense_team_name] += points
        play_log += f" | Score: {score_dict[offense_team_name]}-{score_dict[defense_team_name]}"

    shooter['stats']['possessions'] += 1
    shooter['fatigue'] += 1 + (2 if play_type=='transition' else 0)
    for p in def_court: p['fatigue'] += 0.2

    log.append(play_log)
    return points, possession_time, rebound_team, play_type

# -------------------------------
# Full game simulation
# -------------------------------
def simulate_game(home_team_name, home_roster, away_team_name, away_roster, verbose=False):
    home = copy.deepcopy(home_roster)
    away = copy.deepcopy(away_roster)
    home_starters, home_bench = home[:5], home[5:]
    away_starters, away_bench = away[:5], away[5:]

    for p in home+away:
        p['fatigue']=0; p['fouls']=0; p['disqualified']=False; p['sit_until']=0
        for k in p['stats']: p['stats'][k]=0

    score = {home_team_name:0, away_team_name:0}
    team_fouls={home_team_name:0, away_team_name:0, 'score_diff':0}
    possession_team=home_team_name
    time_left=GAME_SECONDS
    log=[]
    possessions=0
    team_possessions={home_team_name:0, away_team_name:0}
    coach_state={home_team_name:{'style':'balanced','aggressive':False}, away_team_name:{'style':'balanced','aggressive':False}}

    while time_left>0:
        for tname in (home_team_name, away_team_name):
            other = away_team_name if tname==home_team_name else home_team_name
            lead = score[tname]-score[other]
            coach_state[tname]['aggressive'] = lead<-COACH_AGGRESSIVE_DEFICIT and time_left<=COACH_AGGRESSIVE_MINUTES_LEFT*60

        if possession_team==home_team_name:
            off_court = home_starters; def_court = away_starters; is_home=True
            offense_name=home_team_name; defense_name=away_team_name
        else:
            off_court = away_starters; def_court = home_starters; is_home=False
            offense_name=away_team_name; defense_name=home_team_name

        team_fouls['score_diff'] = score[offense_name]-score[defense_name]
        pts, dur, rebound_team, play_type = simulate_possession(
            off_court, def_court, time_left, team_fouls, is_home, log,
            offense_name, defense_name, 1.0,
            coach_aggressive=coach_state[offense_name]['aggressive'],
            score_dict=score
        )

        score[possession_team]+=pts
        team_possessions[possession_team]+=1
        possessions +=1
        time_left -= dur
        if time_left<0: time_left=0

        if possessions % SUB_CHECK_EVERY_POSSESSIONS == 0:
            perform_subs(home_starters, home_bench, home_team_name, time_left, score[home_team_name]-score[away_team_name], log=log)
            perform_subs(away_starters, away_bench, away_team_name, time_left, score[away_team_name]-score[home_team_name], log=log)

        if rebound_team is None or rebound_team==def_court:
            possession_team = away_team_name if possession_team==home_team_name else home_team_name

    return {'score':score, 'players':{p['name']:p['stats'] for p in home+away}, 'log':log, 'team_possessions':team_possessions}

# -------------------------------
# Multi-game averaging
# -------------------------------
def average_player_stats(games, team_name):
    all_players = {}
    total_possessions = 0
    for g in games:
        total_possessions += g['team_possessions'][team_name]
        for pname, stats in g['players'].items():
            if pname not in all_players:
                all_players[pname] = {k:[] for k in stats}
            for k,v in stats.items():
                all_players[pname][k].append(v)
    avg_stats = {p:{k:mean(v) for k,v in stats.items()} for p,stats in all_players.items()}
    totals = {'possessions':total_possessions,
              'team_off_reb':sum(p['off_reb'] for p in avg_stats.values()),
              'team_def_reb':sum(p['def_reb'] for p in avg_stats.values()),
              'team_fg':sum(p['fgm'] for p in avg_stats.values()),
              'team_fga':sum(p['fga'] for p in avg_stats.values())}
    return avg_stats, totals

# -------------------------------
# Print advanced box with +/- adjustment
# -------------------------------
def print_advanced_box_with_plus_minus(team_name, avg_player_stats, opponent_stats, totals, top_n=None):
    player_summary = []
    team_off_pts_allowed = sum(s.get('points',0) for s in opponent_stats.values())
    team_poss = totals.get('possessions', 1)
    pace_factor = 100 / team_poss if team_poss>0 else 1

    for pname, stats in avg_player_stats.items():
        pts = stats.get('points', 0)
        off_reb = stats.get('off_reb', 0)
        def_reb = stats.get('def_reb', 0)
        ast = stats.get('assists', 0)
        stl = stats.get('steals', 0)
        blk = stats.get('blocks', 0)
        tov = stats.get('turnovers', 0)
        fgm = stats.get('fgm', 0)
        fga = stats.get('fga', 1)
        ftm = stats.get('ftm', 0)
        fta = stats.get('fta', 1)
        poss = stats.get('possessions', 1)

        eff = pts + off_reb + def_reb + ast + stl + blk - tov
        usage_pct = (poss / totals.get('possessions', 1)) * 100
        off_reb_pct = (off_reb / max(1, totals.get('team_off_reb', 1))) * 100
        def_reb_pct = (def_reb / max(1, totals.get('team_def_reb', 1))) * 100
        ast_pct = (ast / max(1, fgm)) * 100 if fgm>0 else 0
        tov_pct = (tov / max(1, poss)) * 100

        efg = (fgm + 0.5 * stats.get('3pm',0)) / max(1, fga)
        ts = pts / (2 * max(1, fga + 0.44 * fta))
        def_impact = (stl + blk + def_reb) * 0.5
        player_plus_minus = (pts + off_reb + ast + def_impact - (team_off_pts_allowed*poss/team_poss)) * pace_factor/100

        player_summary.append({
            'player': pname,
            'PTS': pts,
            'ORB': off_reb,
            'DRB': def_reb,
            'AST': ast,
            'STL': stl,
            'BLK': blk,
            'TOV': tov,
            'FG%': round(fgm/fga if fga>0 else 0, 3),
            'FT%': round(ftm/fta if fta>0 else 0, 3),
            'eFG%': round(efg,3),
            'TS%': round(ts,3),
            'USG%': round(usage_pct,1),
            'ORB%': round(off_reb_pct,1),
            'DRB%': round(def_reb_pct,1),
            'AST%': round(ast_pct,1),
            'TOV%': round(tov_pct,1),
            'EFF': round(eff,1),
            '+/-': round(player_plus_minus,1)
        })

    player_summary = sorted(player_summary, key=lambda x: x['EFF'], reverse=True)
    if top_n:
        player_summary = player_summary[:top_n]

    headers = ['Player','PTS','ORB','DRB','AST','STL','BLK','TOV','FG%','FT%','eFG%','TS%','USG%','ORB%','DRB%','AST%','TOV%','EFF','+/-']
    print(f"\nADVANCED BOX (+/-) ({team_name})")
    print("-"*130)
    print("{:<12} {:>3} {:>3} {:>3} {:>3} {:>3} {:>3} {:>3} {:>4} {:>4} {:>4} {:>4} {:>5} {:>4} {:>4} {:>4} {:>4} {:>4} {:>5}".format(*headers))
    print("-"*130)

    for p in player_summary:
        print("{:<12} {:>3} {:>3} {:>3} {:>3} {:>3} {:>3} {:>3} {:>4} {:>4} {:>4} {:>4} {:>5} {:>4} {:>4} {:>4} {:>4} {:>4} {:>5}".format(
            p['player'], p['PTS'], p['ORB'], p['DRB'], p['AST'], p['STL'], p['BLK'], p['TOV'],
            p['FG%'], p['FT%'], p['eFG%'], p['TS%'], p['USG%'], p['ORB%'], p['DRB%'], p['AST%'], p['TOV%'], p['EFF'], p['+/-']
        ))

    print("-"*130)
    team_pts = sum(s['PTS'] for s in player_summary)
    team_off_reb = sum(s['ORB'] for s in player_summary)
    team_def_reb = sum(s['DRB'] for s in player_summary)
    team_ast = sum(s['AST'] for s in player_summary)
    team_stl = sum(s['STL'] for s in player_summary)
    team_blk = sum(s['BLK'] for s in player_summary)
    team_tov = sum(s['TOV'] for s in player_summary)
    team_plus_minus = (team_pts + team_off_reb + team_ast + team_stl + team_blk - team_off_pts_allowed) * pace_factor/100

    print("{:<12} {:>3} {:>3} {:>3} {:>3} {:>3} {:>3} {:>3} {:>4} {:>4} {:>4} {:>4} {:>5} {:>4} {:>4} {:>4} {:>4} {:>4} {:>5}".format(
        f"{team_name} (Team)",
        round(team_pts*pace_factor/100),
        round(team_off_reb*pace_factor/100),
        round(team_def_reb*pace_factor/100),
        round(team_ast*pace_factor/100),
        round(team_stl*pace_factor/100),
        round(team_blk*pace_factor/100),
        round(team_tov*pace_factor/100),
        0,0,0,0,0,0,0,0,0,0,
        round(team_plus_minus,1)
    ))
    print("-"*130)

# -------------------------------
# Monte Carlo simulation
# -------------------------------
def run_simulation(home_team_name, home_roster, away_team_name, away_roster, num_games=50, verbose=False):
    all_games=[]
    for _ in range(num_games):
        g = simulate_game(home_team_name, home_roster, away_team_name, away_roster, verbose=verbose)
        all_games.append(g)
    avg_home_stats, home_totals = average_player_stats(all_games, home_team_name)
    avg_away_stats, away_totals = average_player_stats(all_games, away_team_name)
    print_advanced_box_with_plus_minus(home_team_name, avg_home_stats, avg_away_stats, home_totals, top_n=None)
    print_advanced_box_with_plus_minus(away_team_name, avg_away_stats, avg_home_stats, away_totals, top_n=None)
    avg_home_score = mean([g['score'][home_team_name] for g in all_games])
    avg_away_score = mean([g['score'][away_team_name] for g in all_games])
    print(f"\nAverage final score ({num_games} games): {home_team_name} {avg_home_score:.1f} - {away_team_name} {avg_away_score:.1f}")
    return all_games

# -------------------------------
# Example usage
# -------------------------------
if __name__ == "__main__":
    # Simple 10-player rosters
    home_roster = [create_player(f"Home{i+1}") for i in range(10)]
    away_roster = [create_player(f"Away{i+1}") for i in range(10)]
    run_simulation("Lakers", home_roster, "Celtics", away_roster, num_games=20)



ADVANCED BOX (+/-) (Lakers)
----------------------------------------------------------------------------------------------------------------------------------
Player       PTS ORB DRB AST STL BLK TOV  FG%  FT% eFG%  TS%  USG% ORB% DRB% AST% TOV%  EFF   +/-
----------------------------------------------------------------------------------------------------------------------------------
Home3        7.35 0.7 2.4 0.15   0   0 10.35 0.358 0.852 0.419 0.46   0.5  3.8  4.8  5.7 54.2  0.2   0.0
Away10       4.9 0.1 4.1 0.8   0   0 9.9 0.363 0.65 0.417 0.442   0.4  0.5  8.2 43.2 61.9  0.0   0.0
Home2        5.35 2.35 1.2 0.75   0   0 10.3 0.262 0.895 0.31 0.349   0.5 12.9  2.4 39.5 55.7 -0.7   0.0
Away6        3.2 0.6   6 0.95   0   0 11.5 0.277  1.0 0.313 0.362   0.4  3.3 12.0 82.6 70.8 -0.8   0.0
Away4        5.05 0.45 2.85 0.4   0   0 10.5 0.261 0.778 0.299 0.346   0.5  2.5  5.7 22.9 56.6 -1.8   0.0
Home4        4.7 0.8 4.1 0.45   0   0 12.25 0.344 0.682 0.411 0.445   0.5  4.4  8.2 27.3 67

In [21]:
import random
import copy
from statistics import mean

# -------------------------------
# Config / Tunables
# -------------------------------
GAME_MINUTES = 48
SECONDS_PER_MIN = 60
GAME_SECONDS = GAME_MINUTES * SECONDS_PER_MIN
SUB_CHECK_EVERY_POSSESSIONS = 4
FATIGUE_PLAY_PENALTY = 0.006
FATIGUE_REB_PENALTY = 0.03
COACH_AGGRESSIVE_DEFICIT = 8
COACH_AGGRESSIVE_MINUTES_LEFT = 6

PLAYTYPE_BASE = {
    'transition': 0.10,
    'pnr': 0.30,
    'iso': 0.20,
    'spotup': 0.25,
    'post': 0.15
}

# -------------------------------
# Player creation
# -------------------------------
def create_player(name, role='wing', mpg=30, stats_input=None):
    if stats_input is None: stats_input = {}
    return {
        'name': name,
        'role': role,
        'mpg': mpg,
        '2PA': stats_input.get('2PA', random.uniform(5,15)),
        '2P%': stats_input.get('2P%', random.uniform(0.45,0.55)),
        '3PA': stats_input.get('3PA', random.uniform(0,6)),
        '3P%': stats_input.get('3P%', random.uniform(0.33,0.38)),
        'FTA': stats_input.get('FTA', random.uniform(1,5)),
        'FT%': stats_input.get('FT%', random.uniform(0.7,0.9)),
        'ORB': stats_input.get('ORB', random.uniform(0.5,3)),
        'DRB': stats_input.get('DRB', random.uniform(1.5,8)),
        'AST': stats_input.get('AST', random.uniform(1,8)),
        'STL': stats_input.get('STL', random.uniform(0,2)),
        'BLK': stats_input.get('BLK', random.uniform(0,2)),
        'TO': stats_input.get('TO', random.uniform(1,4)),
        'PF': stats_input.get('PF', random.uniform(1,4)),
        'usage': stats_input.get('usage', random.uniform(15,30)),
        'fatigue': 0.0,
        'fouls': 0,
        'disqualified': False,
        'sit_until': 0,
        'stats': {k:0 for k in ['points','fouls','possessions','assists','off_reb','def_reb',
                                'steals','blocks','turnovers','fgm','fga','3pm','3pa','ftm','fta']}
    }

# -------------------------------
# Helper: swap players
# -------------------------------
def swap_players(on_court, bench, idx_on, bench_player):
    player_out = on_court[idx_on]
    if bench_player not in bench: return False
    bench.remove(bench_player)
    bench.append(player_out)
    on_court[idx_on] = bench_player
    return True

# -------------------------------
# Substitution logic
# -------------------------------
def perform_subs(starters, bench, team_name, game_seconds_left, score_diff, coach_style='balanced', log=None):
    subs_made = 0
    for i, p in enumerate(list(starters)):
        if p.get('disqualified', False) or p['fouls'] >= 6:
            eligible = [b for b in bench if not b.get('disqualified', False) and b.get('sit_until',0)<=game_seconds_left]
            if eligible:
                sub = max(eligible, key=lambda x: x['usage'])
                swap_players(starters, bench, i, sub)
                sub['fatigue'] = max(0, sub['fatigue']-3)
                subs_made += 1
                if log: log.append(f"{team_name} - Sub for fouled out: {p['name']} -> {sub['name']}")
        elif p['fatigue'] > 25 or (p['fatigue']>18 and coach_style=='balanced'):
            eligible = [b for b in bench if not b.get('disqualified', False) and b.get('sit_until',0)<=game_seconds_left]
            if eligible:
                sub = min(eligible, key=lambda x:x['fatigue'])
                swap_players(starters, bench, i, sub)
                subs_made += 1
                if log: log.append(f"{team_name} - Sub for fatigue: {p['name']} -> {sub['name']}")
    return subs_made

# -------------------------------
# Rebound logic
# -------------------------------
def get_rebound(off_court, def_court, is_offensive, log, team_name):
    if is_offensive:
        weights = [max(0.01, p['ORB'] - p['fatigue']*FATIGUE_REB_PENALTY) for p in off_court]
        rebounder = random.choices(off_court, weights=weights, k=1)[0]
        rebounder['stats']['off_reb'] += 1
        log.append(f"{team_name} - {rebounder['name']} grabbed offensive rebound")
    else:
        weights = [max(0.01, p['DRB'] - p['fatigue']*FATIGUE_REB_PENALTY) for p in def_court]
        rebounder = random.choices(def_court, weights=weights, k=1)[0]
        rebounder['stats']['def_reb'] += 1
        log.append(f"{team_name} - {rebounder['name']} grabbed defensive rebound")
    return rebounder

# -------------------------------
# Play type selection
# -------------------------------
def pick_play_type(team, opponent, is_transition, game_seconds_left, score_diff, coach_aggressive=False):
    probs = PLAYTYPE_BASE.copy()
    if is_transition: probs = {k: v*(1.5 if k=='transition' else 0.5) for k,v in probs.items()}
    if coach_aggressive and game_seconds_left<=COACH_AGGRESSIVE_MINUTES_LEFT*60 and score_diff<-COACH_AGGRESSIVE_DEFICIT:
        probs['transition']+=0.05; probs['iso']+=0.05; probs['spotup']+=0.03
    total=sum(probs.values())
    for k in probs: probs[k]/=total
    choices, weights = zip(*probs.items())
    return random.choices(choices, weights=weights, k=1)[0]

def shot_modifier_by_play(play_type):
    return {
        'transition': {'2P_mult':1.05,'3P_mult':0.9,'rebound_effort':0.9},
        'pnr': {'2P_mult':1.02,'3P_mult':1.03,'rebound_effort':1.0},
        'iso': {'2P_mult':0.95,'3P_mult':1.05,'rebound_effort':0.95},
        'spotup': {'2P_mult':0.95,'3P_mult':1.10,'rebound_effort':0.9},
        'post': {'2P_mult':1.10,'3P_mult':0.7,'rebound_effort':1.1}
    }.get(play_type, {'2P_mult':1.0,'3P_mult':1.0,'rebound_effort':1.0})

# -------------------------------
# Single possession simulation
# -------------------------------
def simulate_possession(off_court, def_court, game_seconds_left, team_fouls, is_home, log,
                        offense_team_name, defense_team_name, team_def_rating, coach_aggressive=False,
                        score_dict=None):
    is_transition = random.random()<0.08
    shooter=random.choices(off_court, weights=[p['usage'] for p in off_court], k=1)[0]
    play_type=pick_play_type(off_court, def_court, is_transition, game_seconds_left, team_fouls.get('score_diff',0), coach_aggressive)
    play_mod=shot_modifier_by_play(play_type)
    fatigue_penalty=min(0.25, shooter['fatigue']*FATIGUE_PLAY_PENALTY)
    home_bonus = 0.02 if is_home else 0.0

    base_two_weight = 0.50 * play_mod['2P_mult']
    base_three_weight = (0.20 + 0.5*shooter['3P%']) * play_mod['3P_mult']
    if shooter['role']=='big': base_three_weight*=0.5
    weights=[max(0.01, base_two_weight), max(0.01, base_three_weight), max(0.01, shooter['TO']), max(0.01,0.14)]
    outcome=random.choices(['2P','3P','TO','Foul'], weights=weights,k=1)[0]

    points=0
    rebound_team=None
    play_log=""

    if outcome in ['2P','3P']:
        made = random.random() < min(0.95, max(0.03, (shooter['2P%'] if outcome=='2P' else shooter['3P%'])*play_mod['2P_mult'] - fatigue_penalty + home_bonus))
        shooter['stats']['fga']+=1
        if outcome=='3P': shooter['stats']['3pa']+=1
        if made:
            pts = 2 if outcome=='2P' else 3
            shooter['stats']['points']+=pts
            shooter['stats']['fgm']+=1
            if outcome=='3P': shooter['stats']['3pm']+=1
            points += pts
            play_log=f"{offense_team_name} - {shooter['name']} made {outcome} [{play_type}]"
        else:
            is_offensive_reb = random.random() < 0.28*play_mod['rebound_effort']
            rebound_team = off_court if is_offensive_reb else def_court
            get_rebound(off_court, def_court, is_offensive_reb, log, offense_team_name if is_offensive_reb else defense_team_name)
            play_log=f"{offense_team_name} - {shooter['name']} missed {outcome} [{play_type}]"
    elif outcome=='TO':
        shooter['stats']['turnovers']+=1
        rebound_team=def_court
        play_log=f"{offense_team_name} - {shooter['name']} committed a turnover"
    elif outcome=='Foul':
        defender=random.choice(def_court)
        defender['fouls']+=1
        defender['stats']['fouls']=defender['fouls']
        shooter['stats']['fta']+=1
        ft_made = int(random.random()<shooter['FT%'])
        shooter['stats']['ftm']+=ft_made
        shooter['stats']['points']+=ft_made
        points+=ft_made
        rebound_team=off_court
        play_log=f"{defense_team_name} - {defender['name']} fouled {shooter['name']}; FT made: {ft_made}"

    if score_dict is not None:
        score_dict[offense_team_name] += points
        play_log += f" | Score: {score_dict[offense_team_name]}-{score_dict[defense_team_name]}"

    shooter['stats']['possessions']+=1
    shooter['fatigue']+=1 + (2 if play_type=='transition' else 0)
    for p in def_court: p['fatigue']+=0.2
    log.append(play_log)
    return points, random.randint(6,16), rebound_team, play_type

# -------------------------------
# Simulate full game
# -------------------------------
def simulate_game(home_team_name, home_roster, away_team_name, away_roster, verbose=False):
    home = copy.deepcopy(home_roster)
    away = copy.deepcopy(away_roster)
    home_starters, home_bench = home[:5], home[5:]
    away_starters, away_bench = away[:5], away[5:]

    for p in home+away:
        p['fatigue']=0; p['fouls']=0; p['disqualified']=False; p['sit_until']=0
        for k in p['stats']: p['stats'][k]=0

    score = {home_team_name:0, away_team_name:0}
    team_fouls={home_team_name:0, away_team_name:0, 'score_diff':0}
    possession_team=home_team_name
    time_left=GAME_SECONDS
    log=[]
    possessions=0
    team_possessions={home_team_name:0, away_team_name:0}
    coach_state={home_team_name:{'style':'balanced','aggressive':False}, away_team_name:{'style':'balanced','aggressive':False}}

    while time_left>0:
        for tname in (home_team_name, away_team_name):
            other = away_team_name if tname==home_team_name else home_team_name
            coach_state[tname]['aggressive'] = (score[tname]-score[other]<-COACH_AGGRESSIVE_DEFICIT and time_left<=COACH_AGGRESSIVE_MINUTES_LEFT*60)

        off_court, def_court, is_home, offense_name, defense_name = \
            (home_starters, away_starters, True, home_team_name, away_team_name) \
            if possession_team==home_team_name else (away_starters, home_starters, False, away_team_name, home_team_name)

        team_fouls['score_diff']=score[offense_name]-score[defense_name]

        pts, dur, rebound_team, play_type = simulate_possession(
            off_court, def_court, time_left, team_fouls, is_home, log,
            offense_name, defense_name, 1.0, coach_state[offense_name]['aggressive'], score
        )

        team_possessions[possession_team]+=1
        possessions+=1
        time_left-=dur

        if possessions % SUB_CHECK_EVERY_POSSESSIONS==0:
            perform_subs(home_starters, home_bench, home_team_name, time_left, score[home_team_name]-score[away_team_name], log=log)
            perform_subs(away_starters, away_bench, away_team_name, time_left, score[away_team_name]-score[home_team_name], log=log)

        if rebound_team is None or rebound_team==def_court:
            possession_team = away_team_name if possession_team==home_team_name else home_team_name

    return {'score':score, 'players':{p['name']:p['stats'] for p in home+away}, 'log':log, 'team_possessions':team_possessions}

# -------------------------------
# Summarize log nicely
# -------------------------------
def summarize_possession_log(game_result):
    print("\nPOSSESSION SUMMARY")
    print("-"*90)
    print("{:<5} {:<12} {:<12} {:<12} {:<6} {:<12}".format("No","Team","Shooter","PlayType","PTS","Notes"))
    print("-"*90)
    for i,line in enumerate(game_result['log'],1):
        try:
            parts=line.split(" - ")
            team=parts[0]
            rest=parts[1]
            shooter=rest.split()[0]
            pts=0
            if "made 2P" in rest: pts=2
            elif "made 3P" in rest: pts=3
            elif "FT made: 1" in rest: pts=1
            play_type=rest.split("[")[1].split("]")[0] if "[" in rest else ""
            notes=rest.split(";")[-1] if ";" in rest else ""
        except:
            team=shooter=play_type=notes=""
            pts=0
        print("{:<5} {:<12} {:<12} {:<12} {:<6} {:<12}".format(i,team,shooter,play_type,pts,notes))
    print("-"*90)

# -------------------------------
# Run simulation helper
# -------------------------------
def run_simulation(home_team_name, away_team_name, home_roster, away_roster, n_games=1, verbose=False):
    if n_games==1:
        game_result=simulate_game(home_team_name, home_roster, away_team_name, away_roster)
        if verbose: 
            print(f"\nFinal Score: {home_team_name} {game_result['score'][home_team_name]} - {away_team_name} {game_result['score'][away_team_name]}")
            summarize_possession_log(game_result)
        return game_result
    else:
        results=[]
        for i in range(n_games):
            result=simulate_game(home_team_name, home_roster, away_team_name, away_roster)
            results.append(result)
            if verbose:
                print(f"Game {i+1}: {home_team_name} {result['score'][home_team_name]} - {away_team_name} {result['score'][away_team_name]}")
        avg_home=sum(r['score'][home_team_name] for r in results)/n_games
        avg_away=sum(r['score'][away_team_name] for r in results)/n_games
        print(f"\nAverage over {n_games} games: {home_team_name} {avg_home:.1f} - {away_team_name} {avg_away:.1f}")
        return results

# -------------------------------
# Example dummy rosters
# -------------------------------
home_roster=[create_player(f"LAL_{r}", role=random.choice(['pg','wing','big'])) for r in ["PG","W1","W2","B1","B2","BEN1","BEN2","BEN3","BEN4","BEN5"]]
away_roster=[create_player(f"BOS_{r}", role=random.choice(['pg','wing','big'])) for r in ["PG","W1","W2","B1","B2","BEN1","BEN2","BEN3","BEN4","BEN5"]]

# -------------------------------
# Run examples
# -------------------------------
result = run_simulation("LAL","BOS",home_roster,away_roster,n_games=20,verbose=True)


Game 1: LAL 28 - BOS 37
Game 2: LAL 44 - BOS 41
Game 3: LAL 30 - BOS 31
Game 4: LAL 21 - BOS 21
Game 5: LAL 38 - BOS 33
Game 6: LAL 34 - BOS 28
Game 7: LAL 34 - BOS 31
Game 8: LAL 45 - BOS 24
Game 9: LAL 34 - BOS 19
Game 10: LAL 29 - BOS 36
Game 11: LAL 44 - BOS 38
Game 12: LAL 32 - BOS 40
Game 13: LAL 38 - BOS 43
Game 14: LAL 58 - BOS 28
Game 15: LAL 26 - BOS 36
Game 16: LAL 38 - BOS 23
Game 17: LAL 24 - BOS 23
Game 18: LAL 35 - BOS 30
Game 19: LAL 35 - BOS 32
Game 20: LAL 48 - BOS 26

Average over 20 games: LAL 35.8 - BOS 31.0


In [22]:
import random
import copy

# -------------------------------
# Config / Tunables
# -------------------------------
GAME_MINUTES = 48
SECONDS_PER_MIN = 60
GAME_SECONDS = GAME_MINUTES * SECONDS_PER_MIN
SUB_CHECK_EVERY_POSSESSIONS = 4
FATIGUE_PLAY_PENALTY = 0.006
FATIGUE_REB_PENALTY = 0.03
COACH_AGGRESSIVE_DEFICIT = 8
COACH_AGGRESSIVE_MINUTES_LEFT = 6

PLAYTYPE_BASE = {
    'transition': 0.10,
    'pnr': 0.30,
    'iso': 0.20,
    'spotup': 0.25,
    'post': 0.15
}

# -------------------------------
# Player creation
# -------------------------------
def create_player(name, role='wing', mpg=30, stats_input=None):
    if stats_input is None:
        stats_input = {}
    return {
        'name': name,
        'role': role,
        'mpg': mpg,
        '2PA': stats_input.get('2PA', random.uniform(5,15)),
        '2P%': stats_input.get('2P%', random.uniform(0.45,0.55)),
        '3PA': stats_input.get('3PA', random.uniform(0,6)),
        '3P%': stats_input.get('3P%', random.uniform(0.33,0.38)),
        'FTA': stats_input.get('FTA', random.uniform(1,5)),
        'FT%': stats_input.get('FT%', random.uniform(0.7,0.9)),
        'ORB': stats_input.get('ORB', random.uniform(0.5,3)),
        'DRB': stats_input.get('DRB', random.uniform(1.5,8)),
        'AST': stats_input.get('AST', random.uniform(1,8)),
        'STL': stats_input.get('STL', random.uniform(0,2)),
        'BLK': stats_input.get('BLK', random.uniform(0,2)),
        'TO': stats_input.get('TO', random.uniform(1,4)),
        'PF': stats_input.get('PF', random.uniform(1,4)),
        'usage': stats_input.get('usage', random.uniform(15,30)),
        'fatigue': 0.0,
        'fouls': 0,
        'disqualified': False,
        'sit_until': 0,
        'stats': {
            'points':0,'fouls':0,'possessions':0,'assists':0,'off_reb':0,'def_reb':0,
            'steals':0,'blocks':0,'turnovers':0,'fgm':0,'fga':0,'3pm':0,'3pa':0,'ftm':0,'fta':0
        }
    }

# -------------------------------
# Helper: swap players
# -------------------------------
def swap_players(on_court, bench, idx_on, bench_player):
    player_out = on_court[idx_on]
    if bench_player not in bench:
        return False
    bench.remove(bench_player)
    bench.append(player_out)
    on_court[idx_on] = bench_player
    return True

# -------------------------------
# Substitution logic
# -------------------------------
def perform_subs(starters, bench, team_name, game_seconds_left, score_diff, coach_style='balanced', log=None):
    subs_made = 0
    for i, p in enumerate(list(starters)):
        if p.get('disqualified', False) or p['fouls'] >= 6:
            eligible = [b for b in bench if not b.get('disqualified', False) and b.get('sit_until',0)<=game_seconds_left]
            if eligible:
                sub = max(eligible, key=lambda x: x['usage'])
                swap_players(starters, bench, i, sub)
                sub['fatigue'] = max(0, sub['fatigue']-3)
                subs_made += 1
                if log: log.append(f"{team_name} - Sub for fouled out: {p['name']} -> {sub['name']}")
        elif p['fatigue'] > 25 or (p['fatigue']>18 and coach_style=='balanced'):
            eligible = [b for b in bench if not b.get('disqualified', False) and b.get('sit_until',0)<=game_seconds_left]
            if eligible:
                sub = min(eligible, key=lambda x:x['fatigue'])
                swap_players(starters, bench, i, sub)
                subs_made += 1
                if log: log.append(f"{team_name} - Sub for fatigue: {p['name']} -> {sub['name']}")
        elif coach_style=='conservative' and score_diff>10 and game_seconds_left>5*60:
            eligible = [b for b in bench if b.get('sit_until',0)<=game_seconds_left]
            if eligible:
                sub = min(eligible, key=lambda x:x['fatigue'])
                swap_players(starters, bench, i, sub)
                subs_made += 1
                if log: log.append(f"{team_name} - Sub to rest (leading): {p['name']} -> {sub['name']}")
    return subs_made

# -------------------------------
# Rebound logic
# -------------------------------
def get_rebound(off_court, def_court, is_offensive, log, team_name):
    if is_offensive:
        weights = [max(0.01, p['ORB'] - p['fatigue']*FATIGUE_REB_PENALTY) for p in off_court]
        rebounder = random.choices(off_court, weights=weights, k=1)[0]
        rebounder['stats']['off_reb'] += 1
        log.append(f"{team_name} - {rebounder['name']} grabbed offensive rebound")
    else:
        weights = [max(0.01, p['DRB'] - p['fatigue']*FATIGUE_REB_PENALTY) for p in def_court]
        rebounder = random.choices(def_court, weights=weights, k=1)[0]
        rebounder['stats']['def_reb'] += 1
        log.append(f"{team_name} - {rebounder['name']} grabbed defensive rebound")
    return rebounder

# -------------------------------
# Play type selection
# -------------------------------
def pick_play_type(team, opponent, is_transition, game_seconds_left, score_diff, coach_aggressive=False):
    probs = PLAYTYPE_BASE.copy()
    if is_transition:
        probs = {k: v*(1.5 if k=='transition' else 0.5) for k,v in probs.items()}
    if coach_aggressive and game_seconds_left<=COACH_AGGRESSIVE_MINUTES_LEFT*60 and score_diff<-COACH_AGGRESSIVE_DEFICIT:
        probs['transition']+=0.05; probs['iso']+=0.05; probs['spotup']+=0.03
    total=sum(probs.values())
    for k in probs: probs[k]/=total
    choices, weights = zip(*probs.items())
    return random.choices(choices, weights=weights, k=1)[0]

def shot_modifier_by_play(play_type):
    if play_type=='transition': return {'2P_mult':1.05,'3P_mult':0.9,'rebound_effort':0.9}
    if play_type=='pnr': return {'2P_mult':1.02,'3P_mult':1.03,'rebound_effort':1.0}
    if play_type=='iso': return {'2P_mult':0.95,'3P_mult':1.05,'rebound_effort':0.95}
    if play_type=='spotup': return {'2P_mult':0.95,'3P_mult':1.10,'rebound_effort':0.9}
    if play_type=='post': return {'2P_mult':1.10,'3P_mult':0.7,'rebound_effort':1.1}
    return {'2P_mult':1.0,'3P_mult':1.0,'rebound_effort':1.0}

# -------------------------------
# Simulate a single possession
# -------------------------------
def simulate_possession(off_court, def_court, game_seconds_left, team_fouls, is_home, log,
                        offense_team_name, defense_team_name, coach_aggressive=False, score_dict=None):
    is_transition = random.random() < 0.08
    shooter = random.choices(off_court, weights=[p['usage'] for p in off_court], k=1)[0]
    play_type = pick_play_type(off_court, def_court, is_transition, game_seconds_left,
                               score_diff=team_fouls.get('score_diff', 0), coach_aggressive=coach_aggressive)
    play_mod = shot_modifier_by_play(play_type)
    fatigue_penalty = min(0.25, shooter['fatigue'] * FATIGUE_PLAY_PENALTY)
    home_bonus = 0.02 if is_home else 0.0

    # Outcome weights
    base_two_weight = 0.50 * play_mod['2P_mult']
    base_three_weight = (0.20 + 0.5 * shooter['3P%']) * play_mod['3P_mult']
    if shooter['role'] == 'big': base_three_weight *= 0.5
    turnover_weight = shooter['TO']
    foul_weight = 0.14
    weights = [max(0.01, base_two_weight), max(0.01, base_three_weight),
               max(0.01, turnover_weight), max(0.01, foul_weight)]
    outcome = random.choices(['2P','3P','TO','Foul'], weights=weights, k=1)[0]

    possession_time = random.randint(6, 16)
    points = 0
    rebound_team = None
    play_log = ""

    if outcome in ['2P','3P']:
        base_prob = shooter['2P%'] if outcome == '2P' else shooter['3P%']
        effective_prob = base_prob * (play_mod['2P_mult'] if outcome == '2P' else play_mod['3P_mult']) - fatigue_penalty + home_bonus
        effective_prob = max(0.03, min(0.95, effective_prob))
        made = random.random() < effective_prob
        shooter['stats']['fga'] += 1
        if outcome=='3P': shooter['stats']['3pa']+=1
        if made:
            pts = 2 if outcome=='2P' else 3
            shooter['stats']['fgm'] += 1
            if outcome=='3P': shooter['stats']['3pm']+=1
            shooter['stats']['points'] += pts
            points += pts
            # Possible assist
            eligible_passers = [p for p in off_court if p['name'] != shooter['name']]
            if eligible_passers and random.random()<0.28:
                assister = random.choices(eligible_passers, weights=[p['AST'] for p in eligible_passers], k=1)[0]
                assister['stats']['assists'] += 1
                play_log = f"{offense_team_name} - {shooter['name']} made {outcome} (assisted by {assister['name']}) [{play_type}]"
            else:
                play_log = f"{offense_team_name} - {shooter['name']} made {outcome} [{play_type}]"
        else:
            off_reb_prob = 0.28 * play_mod['rebound_effort']
            is_offensive_reb = random.random() < off_reb_prob
            rebound_team = off_court if is_offensive_reb else def_court
            rebounder = get_rebound(off_court, def_court, is_offensive_reb, log, offense_team_name if is_offensive_reb else defense_team_name)
            possession_time += random.randint(2,5)
            play_log = f"{offense_team_name} - {shooter['name']} missed {outcome} [{play_type}]; rebound by {rebounder['name']}"
    elif outcome=='TO':
        rebound_team = def_court
        possession_time = random.randint(3,8)
        shooter['stats']['turnovers'] += 1
        play_log = f"{offense_team_name} - {shooter['name']} committed a turnover"
    elif outcome=='Foul':
        defender = random.choice(def_court)
        defender['fouls'] += 1
        defender['stats']['fouls'] = defender['fouls']
        shooter['stats']['fta'] += 1
        ft_made = int(random.random() < shooter['FT%'])
        shooter['stats']['ftm'] += ft_made
        shooter['stats']['points'] += ft_made
        points += ft_made
        rebound_team = off_court
        play_log = f"{defense_team_name} - {defender['name']} fouled {shooter['name']}; FT made: {ft_made}"

    # Update score
    if score_dict is not None:
        score_dict[offense_team_name] += points
        play_log += f" | Score: {score_dict[offense_team_name]} - {score_dict[defense_team_name]}"

    shooter['stats']['possessions'] += 1
    shooter['fatigue'] += 1 + (2 if play_type=='transition' else 0)
    for p in def_court: p['fatigue'] += 0.2

    log.append(play_log)
    return points, possession_time, rebound_team, play_type

# -------------------------------
# Simulate full game
# -------------------------------
def simulate_game(home_team_name, home_roster, away_team_name, away_roster, verbose=False):
    home = copy.deepcopy(home_roster)
    away = copy.deepcopy(away_roster)
    home_starters, home_bench = home[:5], home[5:]
    away_starters, away_bench = away[:5], away[5:]

    for p in home+away:
        p['fatigue']=0.0; p['fouls']=0; p['disqualified']=False; p['sit_until']=0
        for k in p['stats']: p['stats'][k]=0

    score = {home_team_name:0, away_team_name:0}
    team_fouls = {home_team_name:0, away_team_name:0, 'score_diff':0}
    possession_team = home_team_name
    time_left = GAME_SECONDS
    log=[]
    possessions=0
    team_possessions = {home_team_name:0, away_team_name:0}
    coach_state = {home_team_name:{'style':'balanced','aggressive':False},
                   away_team_name:{'style':'balanced','aggressive':False}}

    while time_left>0:
        for tname in (home_team_name, away_team_name):
            other = away_team_name if tname==home_team_name else home_team_name
            lead = score[tname]-score[other]
            if lead<-COACH_AGGRESSIVE_DEFICIT and time_left<=COACH_AGGRESSIVE_MINUTES_LEFT*60:
                coach_state[tname]['aggressive']=True
            else: coach_state[tname]['aggressive']=False

        if possession_team==home_team_name:
            off_court, def_court = home_starters, away_starters
            is_home=True; offense_name=home_team_name; defense_name=away_team_name
        else:
            off_court, def_court = away_starters, home_starters
            is_home=False; offense_name=away_team_name; defense_name=home_team_name

        team_fouls['score_diff']=score[offense_name]-score[defense_name]

        pts, dur, rebound_team, play_type = simulate_possession(
            off_court, def_court, time_left, team_fouls, is_home, log,
            offense_name, defense_name, coach_aggressive=coach_state[offense_name]['aggressive'],
            score_dict=score
        )

        team_possessions[possession_team]+=1
        possessions+=1
        time_left-=dur
        if time_left<0: time_left=0

        if possessions%SUB_CHECK_EVERY_POSSESSIONS==0:
            perform_subs(home_starters, home_bench, home_team_name, time_left,
                         score[home_team_name]-score[away_team_name], coach_style='balanced', log=log)
            perform_subs(away_starters, away_bench, away_team_name, time_left,
                         score[away_team_name]-score[home_team_name], coach_style='balanced', log=log)

        if rebound_team is None or rebound_team==def_court:
            possession_team = away_team_name if possession_team==home_team_name else home_team_name

    return {'score':score, 'players':{p['name']:p['stats'] for p in home+away}, 'log':log, 'team_possessions':team_possessions}

# -------------------------------
# Generate advanced box score summary
# -------------------------------
def summarize_box_score(players_stats, team_possessions):
    summary=[]
    for pname, stats in players_stats.items():
        pts=stats['points']
        off_reb=stats['off_reb']
        def_reb=stats['def_reb']
        ast=stats['assists']
        stl=stats['steals']
        blk=stats['blocks']
        tov=stats['turnovers']
        fgm=stats['fgm']; fga=stats['fga']; ftm=stats['ftm']; fta=stats['fta']
        poss=stats['possessions']

        eff=pts+off_reb+def_reb+ast+stl+blk-tov
        usage=(poss/team_possessions)*100 if team_possessions>0 else 0
        off_reb_pct=(off_reb/max(1,sum(p['off_reb'] for p in players_stats.values())))*100
        def_reb_pct=(def_reb/max(1,sum(p['def_reb'] for p in players_stats.values())))*100
        ast_pct=(ast/max(1,fgm))*100
        stl_pct=(stl/max(1,poss))*100
        blk_pct=(blk/max(1,poss))*100
        tov_pct=(tov/max(1,poss))*100

        summary.append({
            'player':pname,'PTS':pts,'FGM':fgm,'FGA':fga,'FTM':ftm,'FTA':fta,
            'ORB':off_reb,'DRB':def_reb,'AST':ast,'STL':stl,'BLK':blk,'TO':tov,
            'Efficiency':eff,'Usage%':usage,'OREB%':off_reb_pct,'DREB%':def_reb_pct,
            'AST%':ast_pct,'STL%':stl_pct,'BLK%':blk_pct,'TO%':tov_pct
        })
    return summary

# -------------------------------
# Run simulation(s)
# -------------------------------
def run_simulation(home_team_name, home_roster, away_team_name, away_roster, num_games=1, verbose=False):
    if num_games==1:
        game = simulate_game(home_team_name, home_roster, away_team_name, away_roster, verbose=verbose)
        box = summarize_box_score(game['players'], sum(game['team_possessions'].values()))
        return {'score':game['score'], 'log':game['log'], 'box_score':box}
    else:
        scores=[]
        accumulated_stats={}
        for g in range(num_games):
            game = simulate_game(home_team_name, home_roster, away_team_name, away_roster, verbose=False)
            scores.append(game['score'])
            for pname, stats in game['players'].items():
                if pname not in accumulated_stats: accumulated_stats[pname]={k:0 for k in stats}
                for k in stats: accumulated_stats[pname][k]+=stats[k]

        # Average stats
        avg_stats={}
        for pname, stats in accumulated_stats.items():
            avg_stats[pname]={k: v/num_games for k,v in stats.items()}

        avg_box = summarize_box_score(avg_stats, sum([sum(s.values()) for s in scores])/num_games)
        avg_score = {home_team_name:sum(s[home_team_name] for s in scores)/num_games,
                     away_team_name:sum(s[away_team_name] for s in scores)/num_games}
        return {'average_score':avg_score, 'scores':scores, 'average_box_score':avg_box}

# -------------------------------
# Example Usage
# -------------------------------
if __name__=="__main__":
    # Create rosters
    home_roster=[create_player(f"Home_{i+1}") for i in range(10)]
    away_roster=[create_player(f"Away_{i+1}") for i in range(10)]

    # Single game example
    result_single = run_simulation("Home", home_roster, "Away", away_roster, num_games=1)
    print("=== Single Game ===")
    print("Final Score:", result_single['score'])
    print("\nPlay-by-Play Log:")
    for p in result_single['log'][:10]: print(p)
    print("\nAdvanced Box Score:")
    for p in result_single['box_score'][:5]: print(p)

    # Monte Carlo example
    result_mc = run_simulation("Home", home_roster, "Away", away_roster, num_games=5)
    print("\n=== Monte Carlo (5 games) ===")
    print("Average Score:", result_mc['average_score'])
    print("All Game Scores:", result_mc['scores'])
    print("\nAverage Advanced Box Score:")
    for p in result_mc['average_box_score'][:5]: print(p)


=== Single Game ===
Final Score: {'Home': 44, 'Away': 42}

Play-by-Play Log:
Home - Home_4 committed a turnover | Score: 0 - 0
Away - Away_2 committed a turnover | Score: 0 - 0
Home - Home_2 committed a turnover | Score: 0 - 0
Away - Away_3 committed a turnover | Score: 0 - 0
Home - Home_4 committed a turnover | Score: 0 - 0
Away - Away_5 committed a turnover | Score: 0 - 0
Home - Home_2 committed a turnover | Score: 0 - 0
Away - Away_5 committed a turnover | Score: 0 - 0
Away - Away_2 grabbed defensive rebound
Home - Home_5 missed 3P [spotup]; rebound by Away_2 | Score: 0 - 0

Advanced Box Score:
{'player': 'Home_1', 'PTS': 6, 'FGM': 2, 'FGA': 7, 'FTM': 1, 'FTA': 1, 'ORB': 1, 'DRB': 1, 'AST': 0, 'STL': 0, 'BLK': 0, 'TO': 8, 'Efficiency': 0, 'Usage%': 4.584527220630372, 'OREB%': 4.761904761904762, 'DREB%': 1.7857142857142856, 'AST%': 0.0, 'STL%': 0.0, 'BLK%': 0.0, 'TO%': 50.0}
{'player': 'Home_2', 'PTS': 2, 'FGM': 1, 'FGA': 3, 'FTM': 0, 'FTA': 0, 'ORB': 2, 'DRB': 3, 'AST': 0, 'STL': 0,